# Scopus Content Service Desk 2026 — validation rules and rejection reasons

Every ticket in the 2026 CSD file was **rejected**: all 56,935 rows carry an `Invalid-*`
resolution code. This notebook connects three artefacts that describe *why* requests get
rejected, none of which is currently linked to actual tickets.

| Artefact | What it gives us |
|---|---|
| `Triangulation - Validation Rules vs Resolution Codes vs Templates.xlsx`, sheet `Triangulation Matrix` | 24 **validation rules** ("invalid when …"), each with a guideline reference, a primary resolution code, a CSD template and a template scenario |
| `scopus content_2026_metadata.tsv` | the **resolution code** the agent actually assigned — ground truth |
| `CSD CheatSheet&Templates.xlsx` | the **response templates** agents paste verbatim into replies |

We produce three things:

1. a **validation rule** tag on every ticket, or `NA` where none of the 24 rules applies;
2. a **rejection reason** taxonomy discovered bottom-up from the tickets themselves, and a
   rejection-reason tag on every ticket;
3. the quantified three-way relationship **rejection reason × validation rule × resolution code**.

## Read this before running anything

**Only 12 of the 15 resolution codes appear as a rule's primary code.** `Invalid- advice request`
(8,070 tickets), `Invalid- book` (592) and `Invalid- discontinued title` (3,989 — an "also maps
to" only) have no rule that claims them. Section 2 computes it exactly: about **78% of tickets carry a code some rule
claims**. Call that the matrix's *code coverage*.

**Be careful what that 78% bounds.** It limits `conformance_strict`, which is measured over every
rule-tagged ticket — a ticket whose code no rule claims can never conform. It does **not** bound
the headline, `conformance among mappable`, which is already restricted to tickets that *can*
conform and whose maximum is therefore 100%. The two numbers can land close together by
coincidence; reading one as the other's ceiling will tell you that you are finished when you still
have all your headroom. Section 18.3 prints each rate next to the bound that actually applies.

**The dataset is entirely rejections, so "rejection reason" is not a niche field — it is the
outcome variable.** The validation rule is the *policy* the agent invoked; the rejection reason is
the *observed ground* stated in the reply. They should agree often, and where they do not, the
matrix has a hole. Section 18.5 finds those holes and Section 16 drafts candidate rules for them.

## Cost and gates

Nothing spends money before Section 7. Each gate below states its cost before it runs.

| Section | What | Cost |
|---|---|---|
| 1–6 | parse the matrix, build prompts, preprocess | free |
| 7 | discover the rejection-reason taxonomy (400 tickets) | ~$3 |
| 11 | 40-ticket smoke test | ~$0.60 |
| 12 | 1,000-ticket conformance probe | ~$14 |
| 14 | main run, 12,000 stratified tickets | ~$156 |
| 16 | candidate-rule pass over the uncovered segment | ~$15 |
| 17 | full corpus (50,721) — **gated off by default** | ~$660 |

## Two things that will bite you

- **`rca build build` rewrites `generated/pipeline_config.json` wholesale.** Running it
  here removes the ScienceDirect 2026 modules from the live config. Section 0c backs them up and
  writes a one-line restore command.
- **Checkpoints resume silently.** Edit a prompt, re-run, and the pipeline logs `todo=0` and makes
  no LLM calls. Section 10 fingerprints every prompt and discards the checkpoint when it changes.


## 0. Configuration — edit this cell

In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

# -- Project root discovery (walk up until rca_pipeline/ is found) ----------
_root = Path().absolute()
while not (_root / "definitions").is_dir() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
BASE = _root

# The system Python has pandas built against NumPy 1.x under NumPy 2.x and
# raises "_ARRAY_API not found" on import. Fail now rather than mid-run.
assert pd.__version__.startswith('3.'), (
    f'pandas {pd.__version__} - expected 3.x from .venv. Wrong kernel selected?')

# -- Inputs ----------------------------------------------------------------
DATA_DIR    = BASE / 'data' / 'scopus content'
TICKETS_CSV = DATA_DIR / 'scopus content_2026_threads_collapsed.csv'
META_TSV    = DATA_DIR / 'scopus content_2026_metadata.tsv'
MATRIX_XLSX = DATA_DIR / 'Triangulation - Validation Rules vs Resolution Codes vs Templates.xlsx'
TEMPLATES_XLSX = DATA_DIR / 'CSD CheatSheet&Templates.xlsx'
SHEET_MATRIX = 'Triangulation Matrix'
SHEET_LEGEND = 'Legend & Method'

# -- Build artefacts -------------------------------------------------------
DEFN_XLSX     = BASE / 'definitions' / 'csd_2026_pipeline_definition.xlsx'
TAXONOMY_DIR  = BASE / 'generated' / 'taxonomies'
DRAFT_DIR     = TAXONOMY_DIR / 'drafts'
MODULES_DIR   = BASE / 'generated' / 'modules'
CONFIG_BACKUP = BASE / 'generated' / '_backup_pre_csd2026'

# -- Columns ---------------------------------------------------------------
ID_COL      = 'INCIDENT_REF'
TEXT_COL    = 'Conversation'
META_ID_COL = 'incident_reference'
RC_COL      = 'resolution_code_lvl1_desc'   # ground truth, never shown to the LLM
CR1_COL     = 'contact_reason_lvl1_desc'
CR2_COL     = 'contact_reason_lvl2_desc'
META_DATE   = 'incident_closed_datetime'

# Safe aliases for the same three fields on any frame derived from a pipeline
# OUTPUT. 'resolution_' and 'contact_reason_' are both in
# rca_pipeline.utils._STRUCTURAL_PREFIXES, so a column keeping its metadata name
# would be silently dropped by drop_enrichment_cols() if that frame were ever fed
# back through the pipeline. Raw corpus frames keep the original names; anything
# derive_conformance() touches uses these.
RC_LABEL    = 'rc_label'
CR1_LABEL   = 'cr1_label'
CR2_LABEL   = 'cr2_label'

# -- Modules ---------------------------------------------------------------
# One module = one taxonomy: rca build bakes a single taxonomy_file into
# each generated .py, so the rule and the reason cannot share a module. They do
# share one profile and one checkpoint (per-module sentinels), so a single
# run_stage() call runs both and resumes each independently.
MOD_RULE   = 'csd_validation_rule'
PREFIX_RULE = 'CSDVR'
STUB_RULE  = 'csdvr'

MOD_REASON  = 'csd_rejection_reason'
PREFIX_REASON = 'CSDRR'
STUB_REASON = 'csdrr'

MOD_PROPOSAL = 'csd_rule_proposal'   # no taxonomy - free-text rule drafting
STUB_PROPOSAL = 'csdprop'

MODULE_PREFIXES = {MOD_RULE: PREFIX_RULE, MOD_REASON: PREFIX_REASON}
TAGGING_MODULES = [MOD_RULE, MOD_REASON]
ALL_MODULES     = TAGGING_MODULES + [MOD_PROPOSAL]

# The code assigned when none of the 24 matrix rules applies. Fixed by row order:
# CSDVR4 is the fourth L1 group and it has exactly one leaf.
NA_RULE_CODE = f'{PREFIX_RULE}4.1'

# -- Preprocessing parameters ---------------------------------------------
# Folded into both modules' fingerprints: changing any of them changes the input
# text without changing a prompt, and the checkpoint keys on record content.
TARGET_YEAR          = 2026
SHORT_CHAR_THRESHOLD = 300      # payload chars below which a ticket is unclassifiable
STUB_PHRASE          = 'transcript of our chat is included'
TRUNC_CAP            = 16_000   # hard ceiling on any conversation after truncation
TRUNC_HEAD           = 4_000    # kept from the start (the customer's original ask)
TRUNC_TAIL           = 11_900   # kept from the end (where the agent's reason lives)
assert TRUNC_HEAD + TRUNC_TAIL + 64 <= TRUNC_CAP

# -- Sampling --------------------------------------------------------------
DISCOVERY_N      = 400      # tickets shown to the taxonomy-discovery pass
DISCOVERY_MIN    = 12       # per resolution code
DISCOVERY_MAX    = 40
SMOKE_N          = 40
PROBE_N          = 1_000
SAMPLE_N         = 12_000   # the main run
SAMPLE_MIN_PER_CODE = 300   # so all 15 codes stay estimable
PROPOSAL_N       = 1_000    # gap pass over the NA segment
RANDOM_SEED      = 0

# -- Outputs ---------------------------------------------------------------
OUT_DIR      = BASE / 'runs' / 'output'
INTER_DIR    = BASE / 'runs' / 'intermediate'
SMOKE_DIR    = BASE / 'runs' / 'smoke_csd_2026'
PROFILE_DIR  = BASE / 'profiles'
CHECKPOINT_DIR = BASE / 'runs' / 'checkpoints'
for _d in (OUT_DIR, INTER_DIR, SMOKE_DIR, PROFILE_DIR, CHECKPOINT_DIR, DRAFT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# Every LLM input stem starts csd_ : run_stage() asserts it, so these checkpoints
# can never collide with the sd_* ones from notebook 07.
STAGE_INPUT    = BASE / 'data' / 'csd_stage_input_2026.csv'
DISCOVERY_CSV  = INTER_DIR / 'csd_discovery_sample.csv'
SAMPLE_CSV     = INTER_DIR / f'csd_sample_{SAMPLE_N}.csv'
EXCLUDED_CSV   = INTER_DIR / 'csd_2026_excluded.csv'
PROPOSAL_CSV   = INTER_DIR / 'csd_proposal_input.csv'
SMOKE_CSV      = SMOKE_DIR / 'csd_smoke_input.csv'
PROBE_CSV      = SMOKE_DIR / 'csd_probe_input.csv'

PROFILE_TAG    = PROFILE_DIR / 'csd_2026_rules_and_reasons.json'
PROFILE_PROP   = PROFILE_DIR / 'csd_2026_rule_proposals.json'

MAIN_OUT       = OUT_DIR / 'csd_2026_rules_and_reasons.xlsx'
PROPOSAL_OUT   = OUT_DIR / 'csd_2026_rule_proposals.xlsx'
FINAL_XLSX     = OUT_DIR / 'csd_2026_validation_rules.xlsx'
FINAL_CSV      = OUT_DIR / 'csd_2026_validation_rules_full_text.csv'

# -- Subprocess environment (UTF-8 matters on Windows) ---------------------
SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONUTF8'] = '1'
SUBPROCESS_ENV['PYTHONIOENCODING'] = 'utf-8'

for _p in (TICKETS_CSV, META_TSV, MATRIX_XLSX, TEMPLATES_XLSX):
    assert _p.exists(), f'Missing: {_p}'
print('Project root :', BASE)
print('Python       :', sys.executable)
print('pandas       :', pd.__version__)
print('Modules      :', ', '.join(ALL_MODULES))
print('Paths OK')

## 0b. Shared helpers

`norm_label` / `norm_prose` / `join_parts` / `slugify` / `leaf_labels` / `read_stage_output` /
`run_cmd` are lifted from [07_sd_2026_two_stage.ipynb](07_sd_2026_two_stage.ipynb) — they are
notebook-local, not library code, so they are copied rather than imported.

`norm_res_code` is new and load-bearing. The resolution codes are written inconsistently across
the three sources: `Invalid- premature` (space after the hyphen) next to `Invalid-Not Missing`
(none), and the matrix has at least one trailing semicolon (`Invalid- reevaluation;`). Any join
between the matrix and the metadata has to go through this function or it will silently produce
zero matches.

`stratified_sample` is also new: every sample in this notebook is stratified by resolution code,
because `Invalid- Out of coverage` has 63 tickets and a plain `df.sample()` would return none of
them.

In [ ]:
_ZERO_WIDTH = dict.fromkeys(map(ord, '​‌‍﻿'))


def norm_label(value) -> str:
    """Normalise a taxonomy/gold label to a stable join key, preserving wording."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    text = str(value)
    if text.strip().lower() in ('', 'nan', 'none'):
        return ''
    text = text.translate(_ZERO_WIDTH)                    # delete before NFKC
    text = unicodedata.normalize('NFKC', text)            # folds U+00A0 -> ' '
    return re.sub(r'\s+', ' ', text).strip()


def norm_prose(value) -> str:
    """Normalise a definition/description cell."""
    return norm_label(value)


def norm_cmp(value) -> str:
    """Case-insensitive comparison key."""
    return norm_label(value).casefold()


_NA_TOKENS = {'', 'na', 'n/a', '[na]', '-', 'none', 'nan', '[n/a]'}


def norm_res_code(value) -> str:
    """Comparison key for a resolution code across the matrix and the metadata.

    Collapses the inconsistent spacing around the hyphen, strips a trailing
    semicolon, and casefolds. Returns '' for the various spellings of 'not
    applicable' so they never join to a real code.
    """
    text = norm_label(value).rstrip(';').strip()
    if text.casefold() in _NA_TOKENS:
        return ''
    text = re.sub(r'^invalid\s*-\s*', 'invalid-', text, flags=re.IGNORECASE)
    return text.casefold()


def split_res_codes(value) -> list:
    """Split an 'Also Maps To' cell into normalised codes, dropping NA markers."""
    parts = re.split(r'[;\n]+', norm_label(value))
    return [c for c in (norm_res_code(p) for p in parts) if c]


def join_parts(*parts) -> str:
    """Join (heading, text) pairs, skipping any with empty text."""
    return ' '.join(f'{head}: {body}' for head, body in parts if body)


def slugify(text) -> str:
    """Deterministic ASCII slug."""
    s = norm_label(text).lower()
    return re.sub(r'_+', '_', re.sub(r'[^a-z0-9]+', '_', s)).strip('_')


def leaf_labels(module: str) -> dict:
    """code -> label for every leaf of a generated taxonomy JSON."""
    tax = json.loads((TAXONOMY_DIR / f'{module}.json').read_text(encoding='utf-8'))
    return {code: leaf['label']
            for node in tax.values()
            for code, leaf in node.get('children', {}).items()}


def read_stage_output(path) -> pd.DataFrame:
    """Read a stage output, keeping the id column as a clean string."""
    path = Path(path)
    df = pd.read_excel(path) if path.suffix.lower().startswith('.xls') else pd.read_csv(path)
    df[ID_COL] = df[ID_COL].astype(str).str.strip()
    return df


def run_cmd(cmd, label):
    """Run a short command, echoing its output; raise on failure."""
    print(f'\n$ {" ".join(str(c) for c in cmd)}')
    r = subprocess.run(cmd, text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
    print(r.stdout or '', end='')
    print(r.stderr or '', end='')
    if r.returncode != 0:
        raise RuntimeError(f'{label} failed (exit {r.returncode})')


def stratified_sample(df: pd.DataFrame, by: str, n: int, *, min_per=0, max_per=None,
                      seed=RANDOM_SEED) -> pd.DataFrame:
    """Sample ~n rows, proportional by `by`, with a floor (and optional ceiling) per stratum.

    Small strata get at least `min_per` rows (or all of them, if fewer exist); the
    remaining budget is shared proportionally. The result is never larger than the
    input and is deterministic for a given seed.
    """
    groups = {k: g for k, g in df.groupby(by, dropna=False)}
    take = {}
    for k, g in groups.items():
        take[k] = min(len(g), min_per)
    remaining = max(0, n - sum(take.values()))
    slack = {k: len(g) - take[k] for k, g in groups.items()}
    total_slack = sum(slack.values())
    if remaining and total_slack:
        for k, g in groups.items():
            extra = int(round(remaining * slack[k] / total_slack))
            take[k] = min(len(g), take[k] + extra)
    if max_per is not None:
        take = {k: min(v, max_per) for k, v in take.items()}
    out = pd.concat(
        [g.sample(n=take[k], random_state=seed) for k, g in groups.items() if take[k]],
        ignore_index=True)
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)


_MARKER_RE = re.compile(r'(?mi)^\s*(customer|agent)(\s+notes)?\s*:')


def payload_len(text) -> int:
    """Conversation length ignoring the speaker markers themselves."""
    return len(_MARKER_RE.sub('', str(text)).strip())


TRUNC_MARKER = '\n\n[...TRUNCATED {n:,} CHARS FROM MIDDLE...]\n\n'


def truncate_middle(text) -> str:
    s = str(text)
    if len(s) <= TRUNC_CAP:
        return s
    cut = len(s) - TRUNC_HEAD - TRUNC_TAIL
    return s[:TRUNC_HEAD] + TRUNC_MARKER.format(n=cut) + s[-TRUNC_TAIL:]


print('Helpers defined.')

### Self-test for the normalisers

`norm_res_code` is the only join key between the matrix and the metadata. If it is wrong,
every conformance number in Section 18 is computed against an empty join and comes out as
zero — which looks like a model failure rather than a bug. Test it before using it.

In [ ]:
_LABEL_CASES = [
    ('  Missing   Content ', 'Missing Content'),
    ('Journal not covered', 'Journal not covered'),
    ('​Premature​', 'Premature'),
    (None, ''), (float('nan'), ''), ('nan', ''),
]
for _raw, _want in _LABEL_CASES:
    _got = norm_label(_raw)
    assert _got == _want, f'norm_label({_raw!r}) -> {_got!r}, expected {_want!r}'

_CODE_CASES = [
    ('Invalid- premature',        'invalid-premature'),
    ('Invalid-premature',         'invalid-premature'),
    ('Invalid -  premature',      'invalid-premature'),
    ('Invalid- reevaluation;',    'invalid-reevaluation'),
    ('Invalid-Not Missing',       'invalid-not missing'),
    ('Invalid- recently delivered in OBII ', 'invalid-recently delivered in obii'),
    ('NA', ''), ('[NA]', ''), ('-', ''), ('', ''), (None, ''),
]
for _raw, _want in _CODE_CASES:
    _got = norm_res_code(_raw)
    assert _got == _want, f'norm_res_code({_raw!r}) -> {_got!r}, expected {_want!r}'

assert split_res_codes('Invalid- Others; Invalid-Captured correct') == \
    ['invalid-others', 'invalid-captured correct']
assert split_res_codes('[NA]') == []
assert split_res_codes(None) == []

# stratified_sample must respect the floor even for a tiny stratum
_toy = pd.DataFrame({'g': ['a'] * 500 + ['b'] * 5, 'v': range(505)})
_s = stratified_sample(_toy, 'g', 100, min_per=10)
assert (_s['g'] == 'b').sum() == 5, 'floor must clamp to the size of a small stratum'
assert len(_s) <= 100 + 10

# payload_len strips the speaker markers themselves, so the stub rule in section 6
# measures what the customer and agent actually wrote and not the scaffolding.
assert payload_len('Customer: hi\nAgent: bye') < len('Customer: hi\nAgent: bye')
assert payload_len('Customer:\nAgent:\nAgent Notes:') == 0
assert payload_len('Customer: hello there') == len(' hello there'.strip())
assert len(truncate_middle('x' * 40_000)) <= TRUNC_CAP + len(TRUNC_MARKER.format(n=0)) + 8

print(f'Self-test passed: {len(_LABEL_CASES) + len(_CODE_CASES)} normaliser cases + '
      f'sampling + truncation.')

## 0c. Safety net — back up the shared config

`rca build build` regenerates `generated/pipeline_config.json` from the enabled
modules of **whichever** definition workbook you pass it
([rca build:618](../rca build)). It does not merge. The live config currently
holds the ScienceDirect 2026 modules from notebook 07; the moment Section 5 runs, they are gone
from it.

The generated `.py` files and taxonomy JSONs survive — only the config that points at them is
replaced — so restoring is a single rebuild. This cell copies the config, the manifest and the
`sd_*` taxonomy JSONs, and writes the restore command next to them. It never overwrites an
existing backup, so re-running the notebook cannot destroy the first one.

In [ ]:
CONFIG_BACKUP.mkdir(parents=True, exist_ok=True)
_backed_up, _skipped = [], []
for _src in [BASE / 'generated' / 'pipeline_config.json',
             BASE / 'generated' / 'build_manifest.json',
             *sorted(TAXONOMY_DIR.glob('sd_*.json'))]:
    if not _src.exists():
        continue
    _dst = CONFIG_BACKUP / _src.name
    if _dst.exists():
        _skipped.append(_src.name)
    else:
        shutil.copy2(_src, _dst)
        _backed_up.append(_src.name)

print(f'Backup dir: {CONFIG_BACKUP.relative_to(BASE)}')
print(f'  copied  : {len(_backed_up)} file(s)')
print(f'  already there (left alone): {len(_skipped)} file(s)')

_live_path = BASE / 'generated' / 'pipeline_config.json'
_live = json.loads(_live_path.read_text(encoding='utf-8'))
print(f'\nModules in the live config right now ({len(_live["modules"])}):')
print('  ' + ', '.join(sorted(_live['modules'])))

_restore = ('rca build --input '
            'definitions/sd_2026_pipeline_definition.xlsx --force-all')
(CONFIG_BACKUP / 'RESTORE.txt').write_text(
    'This notebook (08_csd_2026_validation_rules.ipynb) replaced the ScienceDirect 2026\n'
    'modules in generated/pipeline_config.json with the CSD 2026 ones.\n\n'
    'To put ScienceDirect back:\n\n'
    f'    {_restore}\n\n'
    'The ScienceDirect outputs are already saved under runs/output/sd_2026_*; nothing\n'
    'was deleted. To come back to CSD afterwards, rebuild from\n'
    'definitions/csd_2026_pipeline_definition.xlsx the same way.\n',
    encoding='utf-8')
print(f'\nTo restore ScienceDirect later:\n    {_restore}')

## 1. Read the triangulation matrix

The header sits on row 4 of `Triangulation Matrix` (so `header=3`), the 24 rules occupy the rows
below it, and row 25 is a totals row that must be dropped — it has an empty `#` and would
otherwise become a 25th "rule" with an empty description.

Columns used: **A** `#`, **B** Request Type, **C** Validation Rule, **D** Guideline Ref,
**E** Primary Resolution Code, **F** Also Maps To, **I** CSD Template, **J** Template Scenario,
**K** Mapping Notes / Rationale. Columns **G** and **H** are 2025/2026 resolution-code volumes and
are ignored per the brief.

Columns E and F are the ground truth this notebook tests, so they are read here but **never**
written into the taxonomy the model sees. Columns J and K are — that is how the templates
workbook earns its place: column J names the exact template scenario, and agents paste that
wording verbatim into replies, so it is the single strongest recognition cue available.

The workbook's own `Legend & Method` sheet carries a caveat, printed below, that should travel
with any number this notebook produces.

In [ ]:
_raw_matrix = pd.read_excel(MATRIX_XLSX, sheet_name=SHEET_MATRIX, header=3, dtype=str)
print(f'{SHEET_MATRIX}: {_raw_matrix.shape[0]} rows x {_raw_matrix.shape[1]} cols read')
print('columns:', list(_raw_matrix.columns))
assert _raw_matrix.shape[1] == 11, (
    f'expected 11 columns A..K, got {_raw_matrix.shape[1]} - the sheet layout changed')

COL_NUM, COL_TYPE, COL_RULE, COL_GUIDE, COL_PRIMARY, COL_ALSO = _raw_matrix.columns[:6]
COL_TEMPLATE, COL_SCENARIO, COL_NOTES = _raw_matrix.columns[8:11]

# Drop the totals row: it is the first row with no value in the '#' column.
_has_num = _raw_matrix[COL_NUM].map(norm_label) != ''
_matrix = _raw_matrix[_has_num].copy().reset_index(drop=True)
_dropped = _raw_matrix[~_has_num]
assert len(_matrix) == 24, f'expected 24 rule rows, got {len(_matrix)}'
assert len(_dropped) >= 1, 'expected at least one trailing totals row to drop'
print(f'\n{len(_matrix)} rule rows kept, {len(_dropped)} non-rule row(s) dropped')

RULES = []
for _, r in _matrix.iterrows():
    RULES.append({
        'n':          int(float(norm_label(r[COL_NUM]))),
        'request_type': norm_label(r[COL_TYPE]),
        'rule':       norm_prose(r[COL_RULE]),
        'guideline':  norm_label(r[COL_GUIDE]),
        'primary':    norm_res_code(r[COL_PRIMARY]),
        'primary_raw': norm_label(r[COL_PRIMARY]).rstrip(';').strip(),
        'also':       split_res_codes(r[COL_ALSO]),
        'template':   norm_label(r[COL_TEMPLATE]),
        'scenario':   norm_prose(r[COL_SCENARIO]),
        'notes':      norm_prose(r[COL_NOTES]),
    })

assert [r['n'] for r in RULES] == list(range(1, 25)), 'rule numbers are not 1..24 in order'
assert all(r['rule'] for r in RULES), 'a rule has an empty Validation Rule cell'
assert all(r['primary'] for r in RULES), 'a rule has no primary resolution code'
assert all(r['scenario'] for r in RULES), 'a rule has no Template Scenario (column J)'

_by_type = Counter(r['request_type'] for r in RULES)
print('\nRequest types:')
for _t, _c in _by_type.most_common():
    print(f'  {_c:>2}  {_t}')

print(f'\nDistinct primary codes: {len({r["primary"] for r in RULES})}')
print(f'Rules carrying an "Also Maps To": {sum(1 for r in RULES if r["also"])}')
print(f'Template scenario text: {sum(len(r["scenario"]) for r in RULES):,} chars '
      f'(~{sum(len(r["scenario"]) for r in RULES) // 4:,} tokens) across {len(RULES)} rules')

# The Legend sheet's caveat travels with every number this notebook produces.
_legend = pd.read_excel(MATRIX_XLSX, sheet_name=SHEET_LEGEND, header=None, dtype=str)
_caveats = [norm_prose(v) for v in _legend.to_numpy().ravel()
            if isinstance(v, str) and len(norm_prose(v)) > 80]
print('\n--- Legend & Method (verbatim) ---')
for _c in _caveats[:8]:
    print(f'  {_c}')

## 2. How much of the code vocabulary does the matrix cover?

The point of tagging tickets with a validation rule is to ask whether the rule the ticket shows
maps to the resolution code the agent actually assigned. That question is only answerable for
tickets whose resolution code some rule claims as its **primary** code.

Three high-volume codes are not claimed by any rule:

- `Invalid- advice request` — these are status enquiries ("is journal X indexed?"), not requests.
  All 24 rules are phrased "invalid when …" about a *request*; an enquiry has nothing to
  invalidate. This is the largest single gap and Section 16 drafts candidate rules for it.
- `Invalid- book` — book/conference coverage sits outside the matrix.
- `Invalid- discontinued title` — appears only as an "also maps to" on rule 8.

The workbook itself knows: it has a sheet named `Unmapped 'invalid' res codes`. This cell
computes the coverage from the live data rather than trusting either source.

`MATRIX_CODE_COVERAGE` is a property of **the matrix**, not a performance bound. It caps
`conformance_strict` (measured over all rule-tagged tickets) but says nothing about the headline
`conformance among mappable`, whose maximum is 100%. Section 18.3 keeps them apart.

In [ ]:
_meta_probe = pd.read_csv(META_TSV, sep='\t', dtype=str, usecols=[META_ID_COL, RC_COL])
_rc_counts = _meta_probe[RC_COL].map(norm_label).value_counts()

PRIMARY_CODES = {r['primary'] for r in RULES}
ALSO_CODES    = {c for r in RULES for c in r['also']}

_rows = []
for _label, _n in _rc_counts.items():
    _key = norm_res_code(_label)
    _rows.append({'resolution_code': _label, 'n': int(_n),
                  'is_primary': _key in PRIMARY_CODES,
                  'is_also_only': _key in ALSO_CODES and _key not in PRIMARY_CODES,
                  'unmapped': _key not in PRIMARY_CODES and _key not in ALSO_CODES})
CODE_COVERAGE = pd.DataFrame(_rows).sort_values('n', ascending=False).reset_index(drop=True)
print(CODE_COVERAGE.to_string(index=False))

_total = int(CODE_COVERAGE['n'].sum())
_mappable = int(CODE_COVERAGE.loc[CODE_COVERAGE['is_primary'], 'n'].sum())
MATRIX_CODE_COVERAGE = _mappable / _total

print(f'\nTickets whose resolution code some rule claims as PRIMARY: '
      f'{_mappable:,} / {_total:,}')
print(f'CONFORMANCE CEILING = {MATRIX_CODE_COVERAGE:.1%}   '
      f'({_total - _mappable:,} tickets can never conform)')
print('\nCodes no rule claims as primary:')
for _, _r in CODE_COVERAGE[~CODE_COVERAGE['is_primary']].iterrows():
    _how = 'also-maps-to only' if _r['is_also_only'] else 'not in the matrix at all'
    print(f'  {_r["resolution_code"]:<38} {_r["n"]:>6,}  ({_how})')

# Every primary code in the matrix must exist in the metadata vocabulary, or the
# join in Section 18 silently drops rules.
_vocab = {norm_res_code(v) for v in _rc_counts.index}
_missing = sorted(PRIMARY_CODES - _vocab)
assert not _missing, f'primary codes absent from the metadata vocabulary: {_missing}'

# 'Also maps to' is allowed to name codes the metadata never uses - report, do not fail.
_also_missing = sorted(ALSO_CODES - _vocab)
if _also_missing:
    print(f'\nNote: "Also Maps To" names {len(_also_missing)} code(s) that never occur in '
          f'the 2026 data: {_also_missing}')

assert 0.70 <= MATRIX_CODE_COVERAGE <= 0.85, (
    f'ceiling {MATRIX_CODE_COVERAGE:.3f} is outside the expected band - the matrix or the '
    f'metadata changed shape; re-read Section 1 before trusting anything downstream')

## 3. Load and join the corpus

The collapsed conversations and the ticket metadata are joined here so we can stratify samples by
resolution code and, much later, measure conformance. **The join result never leaves this
notebook's memory.** Section 6 writes the LLM input from an explicit two-column allowlist, so the
model cannot see the answer it is being scored against.

`incident_closed_datetime` in the metadata is 2026 for every row. `Chat_Start_Date` in the
collapsed file is not — 2,011 tickets were opened in late 2024/2025 and closed in 2026. Filtering
on `Chat_Start_Date` would drop those legitimately in-scope tickets, so the year check is an
assertion on the metadata date and `Chat_Start_Date` is carried as a reporting column only.

In [ ]:
tickets_raw = pd.read_csv(TICKETS_CSV, dtype=str, low_memory=False)
tickets_raw[ID_COL] = tickets_raw[ID_COL].astype(str).str.strip()
meta = pd.read_csv(META_TSV, sep='\t', dtype=str, low_memory=False)
meta[META_ID_COL] = meta[META_ID_COL].astype(str).str.strip()

print(f'{TICKETS_CSV.name}: {len(tickets_raw):,} rows x {len(tickets_raw.columns)} cols')
print(f'{META_TSV.name}: {len(meta):,} rows x {len(meta.columns)} cols')
assert tickets_raw[ID_COL].is_unique, 'duplicate INCIDENT_REF in the conversations file'
assert meta[META_ID_COL].is_unique, 'duplicate incident_reference in the metadata'

META_COLS = [META_ID_COL, RC_COL, CR1_COL, CR2_COL, META_DATE,
             'customer_classification_type', 'agent_group_name', 'status_desc']
corpus = tickets_raw.merge(meta[META_COLS], left_on=ID_COL, right_on=META_ID_COL, how='left')
assert len(corpus) == len(tickets_raw), 'the metadata join changed the row count'

_unmatched = int(corpus[RC_COL].isna().sum())
print(f'\nJoined on {ID_COL} == {META_ID_COL}: {len(corpus) - _unmatched:,} matched, '
      f'{_unmatched} unmatched')
assert _unmatched == 0, f'{_unmatched} tickets have no metadata row - cannot score them'

corpus['closed_dt'] = pd.to_datetime(corpus[META_DATE], errors='coerce')
assert corpus['closed_dt'].isna().sum() == 0, 'unparseable incident_closed_datetime'
_years = corpus['closed_dt'].dt.year.value_counts().sort_index()
print(f'\nClosed-date years:\n{_years.to_string()}')
assert set(_years.index) == {TARGET_YEAR}, (
    f'expected every ticket closed in {TARGET_YEAR}; got {sorted(_years.index)}. '
    f'Add an explicit filter here before continuing.')

corpus['rc_key'] = corpus[RC_COL].map(norm_res_code)
assert (corpus['rc_key'] != '').all(), 'a ticket has an empty resolution code'
print(f'\nAll {len(corpus):,} tickets are {TARGET_YEAR}-closed rejections '
      f'({corpus["rc_key"].nunique()} distinct resolution codes).')
print('\nResolution code distribution:')
print(corpus[RC_COL].value_counts().to_string())

## 4. Write the pipeline definition workbook (pass 1 — for discovery)

The Excel workbook is the single source of truth for the pipeline: `rca build` reads the
`modules` / `fields` / `taxonomy` sheets and generates the taxonomy JSONs, the classifier `.py`
files and `pipeline_config.json` from it. Writing it from code rather than by hand means the 24
rule descriptions are guaranteed to match the matrix.

This is **pass 1**. The validation-rule taxonomy is fully populated; the rejection-reason taxonomy
is deliberately empty and its module carries `discovery_mode='full'`, because it does not exist
yet — Section 7 discovers it. Section 9 rewrites this same workbook with the discovered taxonomy
filled in.

The order matters and is not obvious: `rca build discover` reads `discovery_mode`,
`extract_prompt` and the taxonomy prefix from **`generated/pipeline_config.json`**, not from the
Excel. So the sequence has to be *build → discover → write back to Excel → rebuild*. Discovering
before building would silently use a derived prefix and a built-in prompt.

### Taxonomy layout (codes are assigned by row order — this fixes them)

| Group | Leaves | From |
|---|---|---|
| `CSDVR1` Missing Content | `CSDVR1.1` … `CSDVR1.14` | matrix rows 1–14 |
| `CSDVR2` Document Correction | `CSDVR2.1` … `CSDVR2.9` | matrix rows 15–23 |
| `CSDVR3` Document Correction or Missing Content | `CSDVR3.1` | matrix row 24 |
| `CSDVR4` No applicable validation rule | `CSDVR4.1` | authored |

Each leaf description is `Rule: … Guideline: … Agent reply wording: … Notes: …` built from
matrix columns C / D / J / K. Columns E and F never appear.

In [ ]:
# ══ Authored text ════════════════════════════════════════════════════════
# Everything below that is not quoted from the matrix is authored here, so it can
# be reviewed in one place. The rule texts, guideline refs, template scenarios and
# mapping notes come verbatim from Section 1.

# The disambiguation brief. The matrix states each rule in isolation and never says
# how to choose between two that both half-fit, which is the actual failure mode.
RULE_PRECEDENCE = (
    'Decide from the agent reply, not from the customer request. Every ticket in this corpus was '
    'rejected: the customer message tells you what kind of request it was, but only the agent '
    'reply tells you which rule invalidated it. Where the agent names a specific ground - too '
    'recent, already indexed, journal never covered, under re-evaluation, unauthentic publisher '
    'site, article in press, record already matches the version of record - match that ground to '
    'its rule. '
    'Separate rules that look alike by what the agent asserts about the source. A source that was '
    'never selected for Scopus is the not-covered rule; a source that was selected but is '
    'currently on hold is the re-evaluation rule; a source that was covered and no longer is falls '
    'under coverage mismatch. '
    'Separate content that is too recent to be indexed yet from content that is already in the '
    'delivery pipeline by whether the agent says it has been delivered and will appear shortly. '
    'Prefer the no-applicable-rule code over a rule that only half fits.'
)

# The NA leaf. This is load-bearing: roughly one ticket in five belongs here, and
# without an explicit description of what belongs the model force-fits a rule.
NA_RULE_DESC = (
    'Use this code when none of the numbered validation rules applies. This includes: the ticket '
    'is a status or coverage enquiry rather than a request to add or correct content, for example '
    'asking whether a journal is currently indexed, when a title was discontinued, or what the '
    'current status of a source is; the request concerns a book or a conference proceeding '
    'awaiting evaluation; the ticket was routed to another team or is an internal or duplicate '
    'ticket; or the agent gives a ground for refusal that none of the numbered rules describes. '
    'The most common case by far: the agent confirms the current coverage status of a journal, '
    'perhaps adding that any journal may be subject to re-evaluation at any point in time, and '
    'nothing was requested or refused. That is a status enquiry and belongs here, not under the '
    're-evaluation rule. Roughly one ticket in six or seven is expected to land here. Do not '
    'force a rule that does not fit.'
)

# Per-rule NEGATIVE cues, keyed by matrix row number. Appended to the leaf
# description as a separate clause so the quoted matrix text stays verbatim.
#
# Rule 2 exists because the 1,000-ticket probe found the model reading a standing
# disclaimer as a finding. Agents close most coverage-status replies with "any
# journal may be subject to re-evaluation at any point in time"; it appeared in 88
# probe tickets with no source-specific claim, and the model tagged 44% of them as
# re-evaluation. Of the tickets that cited that sentence as their evidence, 23 of 24
# were logged by the agent as advice request and none as reevaluation - so the cue
# is a near-perfect marker of a wrong tag. Left unfixed it also hides the biggest
# gap in the matrix, because those advice requests never reach CSDVR4.1.
RULE_NEGATIVE_CUES = {
    2: ('The sentence any journal may be subject to re-evaluation at any point in time is a '
        'standing disclaimer that closes most coverage-status replies. It is NEVER on its own '
        'sufficient for this rule. If that sentence, or a general remark that coverage is kept '
        'under continuous review, is the only mention of re-evaluation in the ticket, this rule '
        'does not apply - pick the rule that matches what the agent actually refused, or the '
        'no-applicable-rule code if the agent refused nothing. '
        'Discontinued is also not re-evaluation: a title the agent describes as discontinued, '
        'ceased or no longer covered has had a decision taken, so it belongs to the '
        'coverage-mismatch rule, not here. '
        'Use this rule ONLY when the agent says THIS source is currently under re-evaluation, is '
        'on hold pending a decision, or has been referred to the Content Selection and Advisory '
        'Board.'),
    1: ('Not this rule when the agent confirms the source IS currently indexed and the customer '
        'was only asking about its status. Not this rule either when the agent says the source '
        'WAS covered and has since been discontinued or ceased - a title that was indexed and no '
        'longer is belongs to the coverage-mismatch rule, not to never-covered.'),
    5: ('This IS the rule whenever the agent explains the standard indexing timeline in general '
        'terms - for example that customers should allow 6 weeks after the publication date, or '
        'that the indexation process generally takes several weeks for an article to become '
        'visible. That wording is the premature-claim template. It describes how long indexing '
        'normally takes; it does not say anything has been delivered.'),
    8: ('This IS the rule when the agent says the title has been discontinued, has ceased, was '
        'removed from Scopus, or is no longer covered, and the requested article falls after the '
        'last covered year. Discontinued coverage is a coverage mismatch: a decision has already '
        'been taken. Do not route those tickets to the never-covered rule (the title WAS covered) '
        'or to the re-evaluation rule (the decision is not pending).'),
    7: ('Not this rule when the agent only explains the general indexing timeline - allow 6 weeks '
        'after publication, the indexation process generally takes several weeks, the article '
        'should appear in due course. That is the premature rule, not this one. A phrase like '
        'the indexing process is under way describes the normal queue, not a specific delivery. '
        'Use this rule ONLY when the agent states that THIS article has already been captured, '
        'received, delivered or sent for loading and will appear within a few days. If the agent '
        'gives no delivery statement specific to this article, prefer the premature rule.'),
}

RULE_CODE_INSTR = (
    f'Assign the single leaf-level code (for example {PREFIX_RULE}1.5) for the validation rule '
    f'that makes this request invalid. Read the Rule text and the Agent reply wording under each '
    f'code and match on what the ticket actually shows, not on surface keywords. The Agent reply '
    f'wording describes the response template an agent would use for that rule, so it is the '
    f'strongest signal when the agent pasted template text. If no numbered rule applies, use '
    f'{NA_RULE_CODE}. Only use codes from the taxonomy.'
)
RULE_APPLIES_INSTR = (
    f'Answer yes if one of the numbered validation rules genuinely applies to this ticket, and no '
    f'if you assigned the no-applicable-rule code {NA_RULE_CODE}. Answer with a single lowercase '
    f'word.'
)

REASON_CODE_INSTR = (
    f'Assign the single leaf-level code (for example {PREFIX_REASON}1.1) for the reason this '
    f'request was refused. The reason is the ground the agent gave for not actioning the request, '
    f'so read the agent reply. If the agent gave no substantive reason, use the code that means no '
    f'reason was stated. Only use codes from the taxonomy.'
)

JUST_INSTR = (
    'In one or two sentences, explain why this {kind} applies and why the closest alternative '
    'does not. Refer to what the ticket actually shows. Do not use apostrophes or quotation '
    'marks in your answer.'
)
EV_INSTR = (
    'Copy verbatim the single sentence or phrase from the agent reply that most strongly '
    'supports this {kind}. Do not paraphrase, infer, or add words that are not in the '
    'transcript. If no direct evidence exists, return an empty string.'
)

# Pass 1 discovery prompt. The escape hatch in the fourth bullet matters: without it
# the extractor invents a reason for every content-free ticket and Pass 2 builds an
# L1 group out of noise.
REASON_EXTRACT_PROMPT = (
    'You are analysing Scopus Content Service Desk tickets. Every ticket in this corpus was '
    'rejected: the customer request was not actioned. Read the agent reply and identify the '
    'ABSTRACT REASON THE REQUEST WAS REFUSED - the substantive ground the agent gave, not the '
    'customer original request and not the agent tone or process.\n'
    'CRITICAL RULES:\n'
    '- The reason is almost always in the AGENT message, not the customer message.\n'
    '- Merge wording variants into one category. For example too recent, please allow 6 weeks, '
    'indexation takes several weeks, and not yet delivered should ALL become one item: Content '
    'too recently published to be indexed yet.\n'
    '- Name the ground for refusal, not the topic. Good items: Source never selected for Scopus '
    'coverage. Record already matches the publisher version of record. Customer asked about '
    'status only so there was no actionable request. Bad items: Missing article (that is the '
    'request). Agent was polite (that is tone).\n'
    '- If the agent gave no substantive reason, because the ticket is a chat transcript stub, a '
    'handoff, or boilerplate only, return the single item: No reason stated in the ticket. Do not '
    'invent one.\n'
    '- Include reasons that are not about content policy at all where you see them: pure status '
    'enquiries, book or conference coverage, duplicate tickets, requests routed to another team.\n'
    '- Each transcript should produce exactly 1 reason.\n'
    'Return a JSON array - one object per record:\n'
    '[{"uid": "<uid>", "extracted_items": [{"item": "<abstract rejection reason>", "count": 1}]}]\n'
    'Return ONLY valid JSON. No markdown fences.'
)

PROPOSAL_DESC = (
    'You are reviewing Scopus Content Service Desk tickets that were rejected but that none of '
    'the existing 24 content validation rules covers. For each ticket, draft the validation rule '
    'that WOULD cover it, phrased the way the existing rules are phrased: a short condition '
    'describing when a request of this kind is invalid. Base the rule only on what the agent '
    'actually said. Do not restate the customer request, and do not propose a rule that merely '
    'renames an existing one.'
)

print(f'Authored text: precedence {len(RULE_PRECEDENCE)} chars, '
      f'NA description {len(NA_RULE_DESC)} chars, '
      f'extract prompt {len(REASON_EXTRACT_PROMPT)} chars')

In [ ]:
# ══ Sheet builders ═══════════════════════════════════════════════════════
# REASON_TAXONOMY is empty on pass 1 (the taxonomy does not exist yet) and filled
# in by Section 8 before pass 2. Each entry: (l1_label, l1_desc, leaf_label, leaf_desc).
REASON_TAXONOMY: list = []

# L1 groups for the rule taxonomy, in matrix row order. The group a rule belongs to
# is its Request Type; the trailing NA group is authored.
RULE_L1_ORDER = ['Missing Content',
                 'Document Correction',
                 'Document Correction Missing Content']
RULE_L1_DESC = {
    'Missing Content': ('Rules for requests to add content that is missing from Scopus.'),
    'Document Correction': ('Rules for requests to correct an existing Scopus record.'),
    'Document Correction Missing Content': (
        'Rules that apply to either a correction request or a missing-content request.'),
}
NA_L1_LABEL = 'No applicable validation rule'


def rule_leaf_description(rule: dict) -> str:
    """Leaf guidance from matrix columns C / D / J / K, plus any authored negative cue.

    The matrix text is quoted verbatim; the negative cue is a clearly separate
    clause so it is obvious in the prompt which part is policy and which is ours.
    """
    return join_parts(('Rule', rule['rule']),
                      ('Guideline', rule['guideline']),
                      ('Agent reply wording', rule['scenario']),
                      ('Notes', rule['notes']),
                      ('Do NOT use', RULE_NEGATIVE_CUES.get(rule['n'], '')))


def build_modules_sheet() -> pd.DataFrame:
    rows = [{
        'module_name': MOD_RULE,
        'taxonomy_prefix': PREFIX_RULE,
        'taxonomy_file': f'{MOD_RULE}.json',
        'reorder_group': 'A', 'reorder_order': 1, 'enabled': True,
        'discovery_mode': '', 'extract_prompt': '',
        'description': (
            f'Decide which Scopus content validation rule makes this rejected support request '
            f'invalid. There are {len(RULES)} numbered rules taken from the CSD triangulation '
            f'matrix, plus one code for when none of them applies. Each rule carries the rule '
            f'text, its guideline reference, the wording an agent would use when replying under '
            f'that rule, and mapping notes. Assign the single best-fitting LEAF code '
            f'(e.g. {PREFIX_RULE}1.5). ' + RULE_PRECEDENCE),
        'notes': 'Taxonomy generated from the Triangulation Matrix. Runs on every ticket.',
    }, {
        'module_name': MOD_REASON,
        'taxonomy_prefix': PREFIX_REASON,
        'taxonomy_file': f'{MOD_REASON}.json',
        'reorder_group': 'B', 'reorder_order': 2, 'enabled': True,
        'discovery_mode': 'full' if not REASON_TAXONOMY else '',
        'extract_prompt': REASON_EXTRACT_PROMPT,
        'description': (
            'Identify the reason a Scopus Content Service Desk request was refused, using a '
            'taxonomy of rejection reasons discovered from the tickets themselves. The reason is '
            'the ground the agent gave for not actioning the request, which is not the same as '
            'what the customer asked for. Assign the single best-fitting LEAF code '
            f'(e.g. {PREFIX_REASON}1.1).'),
        'notes': 'Taxonomy discovered in section 7 of notebook 08, then written back here.',
    }, {
        'module_name': MOD_PROPOSAL,
        'taxonomy_prefix': '', 'taxonomy_file': '',
        'reorder_group': 'C', 'reorder_order': 3, 'enabled': True,
        'discovery_mode': '', 'extract_prompt': '',
        'description': PROPOSAL_DESC,
        'notes': 'No taxonomy. Runs only on the no-applicable-rule segment (section 16).',
    }]
    return pd.DataFrame(rows)


def build_fields_sheet() -> pd.DataFrame:
    def row(module, name, ftype, desc, instr, validate=False):
        return {'module_name': module, 'field_name': name, 'field_type': ftype,
                'description': desc, 'prompt_instruction': instr, 'in_output': True,
                'explode_list': False, 'validate_enum': validate, 'notes': ''}

    rows = [
        row(MOD_RULE, f'{STUB_RULE}_code', 'code',
            'Assigned validation rule leaf code', RULE_CODE_INSTR, validate=True),
        # Deliberately validate_enum=False: the built-in validator lowercases and
        # blanks non-matching values, which would hide a deviation. This field is a
        # tripwire against csdvr_code == NA_RULE_CODE and is checked in pandas instead.
        row(MOD_RULE, f'{STUB_RULE}_applies', 'enum:yes,no',
            'Tripwire: does a numbered rule genuinely apply', RULE_APPLIES_INSTR),
        row(MOD_RULE, f'{STUB_RULE}_justification', 'str',
            'Why this rule over the nearest alternative',
            JUST_INSTR.format(kind='validation rule')),
        row(MOD_RULE, f'{STUB_RULE}_evidence', 'str',
            'Verbatim supporting quote from the agent reply',
            EV_INSTR.format(kind='validation rule')),

        row(MOD_REASON, f'{STUB_REASON}_code', 'code',
            'Assigned rejection reason leaf code', REASON_CODE_INSTR, validate=True),
        row(MOD_REASON, f'{STUB_REASON}_justification', 'str',
            'Why this reason over the nearest alternative',
            JUST_INSTR.format(kind='rejection reason')),
        row(MOD_REASON, f'{STUB_REASON}_evidence', 'str',
            'Verbatim supporting quote from the agent reply',
            EV_INSTR.format(kind='rejection reason')),

        row(MOD_PROPOSAL, f'{STUB_PROPOSAL}_request_type', 'str',
            'Request type the proposed rule belongs to',
            'Name the request type in two or three words, for example Missing Content, Document '
            'Correction, Coverage Enquiry, Book Coverage. Use an existing type where one fits.'),
        row(MOD_PROPOSAL, f'{STUB_PROPOSAL}_rule_text', 'str',
            'Proposed validation rule, phrased as a condition',
            'Draft the validation rule this ticket would need, in one sentence, phrased as the '
            'condition under which such a request is invalid. Start with the condition, not with '
            'the word invalid. Do not use apostrophes or quotation marks.'),
        row(MOD_PROPOSAL, f'{STUB_PROPOSAL}_expected_code', 'str',
            'Resolution code the agent would plausibly assign',
            'Name the single resolution code an agent would most plausibly assign under this '
            'proposed rule, using the wording of the existing Invalid codes. If you cannot tell, '
            'return an empty string.'),
        row(MOD_PROPOSAL, f'{STUB_PROPOSAL}_evidence', 'str',
            'Verbatim supporting quote from the agent reply',
            EV_INSTR.format(kind='proposed rule')),
    ]
    return pd.DataFrame(rows)


def build_taxonomy_sheet() -> pd.DataFrame:
    rows = []
    # -- validation rules: one L1 per request type, in matrix row order ----
    for l1 in RULE_L1_ORDER:
        rows.append({'module_name': MOD_RULE, 'level': 'L1', 'label': l1,
                     'description': RULE_L1_DESC[l1], 'notes': 'from Request Type'})
        for rule in [r for r in RULES if r['request_type'] == l1]:
            rows.append({'module_name': MOD_RULE, 'level': 'L2',
                         'label': f'{rule["n"]:02d} {rule["rule"][:90]}',
                         'description': rule_leaf_description(rule),
                         'notes': f'matrix row {rule["n"]}'})
    rows.append({'module_name': MOD_RULE, 'level': 'L1', 'label': NA_L1_LABEL,
                 'description': 'The ticket is not covered by any numbered validation rule.',
                 'notes': 'authored'})
    rows.append({'module_name': MOD_RULE, 'level': 'L2', 'label': NA_L1_LABEL,
                 'description': NA_RULE_DESC, 'notes': 'authored'})

    # -- rejection reasons: empty on pass 1, discovered in section 7 -------
    seen_l1 = set()
    for l1_label, l1_desc, leaf_label, leaf_desc in REASON_TAXONOMY:
        if l1_label not in seen_l1:
            rows.append({'module_name': MOD_REASON, 'level': 'L1', 'label': l1_label,
                         'description': l1_desc, 'notes': 'discovered'})
            seen_l1.add(l1_label)
        rows.append({'module_name': MOD_REASON, 'level': 'L2', 'label': leaf_label,
                     'description': leaf_desc, 'notes': 'discovered'})

    return pd.DataFrame(rows, columns=['module_name', 'level', 'label',
                                       'description', 'notes'])


def write_definition_workbook() -> tuple:
    modules_df  = build_modules_sheet()
    fields_df   = build_fields_sheet()
    taxonomy_df = build_taxonomy_sheet()

    # -- shape checks before anything downstream depends on it -------------
    rule_rows = taxonomy_df[taxonomy_df['module_name'] == MOD_RULE]
    got_l1 = int((rule_rows['level'] == 'L1').sum())
    got_l2 = int((rule_rows['level'] == 'L2').sum())
    assert (got_l1, got_l2) == (len(RULE_L1_ORDER) + 1, len(RULES) + 1), (
        f'{MOD_RULE}: expected {len(RULE_L1_ORDER) + 1} L1 / {len(RULES) + 1} L2, '
        f'got {got_l1} / {got_l2}')

    blank = taxonomy_df[(taxonomy_df['level'] == 'L2') &
                        (taxonomy_df['description'].map(norm_prose) == '')]
    assert blank.empty, f'leaf rows with no guidance:\n{blank[["module_name", "label"]]}'

    # Columns E and F of the matrix must never reach the model.
    _tax_text = ' '.join(taxonomy_df['description'].astype(str)).casefold()
    for _rule in RULES:
        assert _rule['primary_raw'].casefold() not in _tax_text, (
            f'resolution code {_rule["primary_raw"]!r} leaked into a taxonomy description - '
            f'that is the ground truth this notebook measures against')

    # _clean_df drops rows whose FIRST column starts with L1/L2/level/module_name/
    # Green/Edit. The first column is module_name and every value starts with csd_,
    # so nothing is dropped - but never reorder the columns to put `level` first.
    assert taxonomy_df.columns[0] == 'module_name'
    assert taxonomy_df['module_name'].str.startswith('csd_').all()

    # Field stubs must not collide with the structural prefixes that
    # utils.drop_enrichment_cols() strips from any re-run input.
    from rca_pipeline.utils import _STRUCTURAL_PREFIXES
    for _f in fields_df['field_name']:
        _bad = [p for p in _STRUCTURAL_PREFIXES if _f.startswith(p)]
        assert not _bad, f'field {_f!r} starts with structural prefix {_bad} and would be dropped'

    # Pass 1 has no rejection-reason taxonomy, and rca build skips writing an
    # empty one - so a JSON left behind by a PREVIOUS complete run would survive and
    # section 5 would build prompts against last run's taxonomy without saying so.
    # Delete it here to make pass 1 idempotent.
    if not REASON_TAXONOMY:
        _stale = TAXONOMY_DIR / f'{MOD_REASON}.json'
        if _stale.exists():
            _stale.unlink()
            print(f'Removed a stale {_stale.name} from an earlier run '
                  f'(pass 1 has no discovered taxonomy yet).')

    DEFN_XLSX.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(DEFN_XLSX, engine='openpyxl') as w:
        modules_df.to_excel(w, sheet_name='modules', index=False)
        fields_df.to_excel(w, sheet_name='fields', index=False)
        taxonomy_df.to_excel(w, sheet_name='taxonomy', index=False)
    return modules_df, fields_df, taxonomy_df


modules_df, fields_df, taxonomy_df = write_definition_workbook()

print(f'Wrote {DEFN_XLSX.relative_to(BASE)}')
print(f'  {len(modules_df)} modules, {len(fields_df)} fields, '
      f'{len(taxonomy_df)} taxonomy rows')
for _m in ALL_MODULES:
    _sub = taxonomy_df[taxonomy_df['module_name'] == _m]
    _l1 = int((_sub['level'] == 'L1').sum())
    _l2 = int((_sub['level'] == 'L2').sum())
    _disc = modules_df.loc[modules_df['module_name'] == _m, 'discovery_mode'].iloc[0]
    print(f'  {_m:<24} {MODULE_PREFIXES.get(_m, "-"):<6} '
          f'{_l1:>2} L1 / {_l2:>2} L2   discovery_mode={_disc!r}')
print(f'\n{MOD_REASON} has an empty taxonomy on purpose - section 7 discovers it.')

## 5. Gate 1 — build the pipeline and read the prompts *(free)*

This is the last point at which everything is still free and fully inspectable. The build turns
the workbook into three artefacts per module — a taxonomy JSON, a generated classifier `.py`, and
an entry in the shared `pipeline_config.json` — and then we reconstruct the exact system prompt
each classifier will send and assert that nothing was lost on the way.

The verbatim check is the one that matters. It asserts that **all 24 rule texts and all 24
template scenarios reach the model**. If a description silently truncated or an Excel cell came
back as `nan`, the model would be classifying against a taxonomy of labels with no guidance and
the accuracy would drop for reasons invisible in the output.

`csd_rejection_reason` has no taxonomy yet, so `write_taxonomy()` skips it and no JSON exists —
this cell handles that and reports it rather than failing. `csd_rule_proposal` has no taxonomy at
all by design, so its generated module carries a flat `_SYSTEM_PROMPT` constant instead of a
`_build_system_prompt()` function.

In [ ]:
run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
         '--input', str(DEFN_XLSX), '--dry-run'], 'build --dry-run')

In [ ]:
run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
         '--input', str(DEFN_XLSX), '--force-all'], 'build --force-all')

_cfg = json.loads((BASE / 'generated' / 'pipeline_config.json').read_text(encoding='utf-8'))
print('\nModules now in the live config:', sorted(_cfg['modules']))
assert set(_cfg['modules']) == set(ALL_MODULES), (
    f'config holds {sorted(_cfg["modules"])}, expected {sorted(ALL_MODULES)}')

# Prefixes must not nest: leaf codes are parsed with a greedy ^[A-Z]+ match, so a
# prefix that is a prefix of another would make CSDVR1.1 ambiguous.
_prefixes = [p for p in MODULE_PREFIXES.values()]
for _a in _prefixes:
    for _b in _prefixes:
        assert _a == _b or not _b.startswith(_a), f'prefix {_a!r} nests inside {_b!r}'

# The sentinel is the field the checkpoint uses to decide a record is done for a
# module. With two modules in one profile, getting this wrong means one module
# never resumes - or worse, is skipped entirely.
sys.path.insert(0, str(BASE))
from rca_pipeline.runner import _build_sentinel_map  # noqa: E402
SENTINELS = _build_sentinel_map(_cfg)
print('\nCheckpoint sentinels:')
for _m in ALL_MODULES:
    print(f'  {_m:<24} -> {SENTINELS.get(_m)!r}')
assert SENTINELS.get(MOD_RULE) == f'{STUB_RULE}_code', SENTINELS.get(MOD_RULE)
assert SENTINELS.get(MOD_REASON) == f'{STUB_REASON}_code', SENTINELS.get(MOD_REASON)

_rule_tax = json.loads((TAXONOMY_DIR / f'{MOD_RULE}.json').read_text(encoding='utf-8'))
_n_l1 = len(_rule_tax)
_n_leaf = sum(len(v.get('children', {})) for v in _rule_tax.values())
print(f'\n{MOD_RULE}.json: {_n_l1} L1 groups, {_n_leaf} leaves')
assert (_n_l1, _n_leaf) == (len(RULE_L1_ORDER) + 1, len(RULES) + 1)
assert NA_RULE_CODE in _rule_tax[f'{PREFIX_RULE}4']['children'], (
    f'{NA_RULE_CODE} is not where the code expects it - the L1 row order changed')
print(f'NA code {NA_RULE_CODE} = '
      f'{_rule_tax[f"{PREFIX_RULE}4"]["children"][NA_RULE_CODE]["label"]!r}')

In [ ]:
import importlib

from rca_pipeline.taxonomy import taxonomy_to_rich_prompt

importlib.invalidate_caches()


def build_prompts() -> dict:
    """Reconstruct the exact system prompt each generated module will send.

    Modules with a taxonomy expose _build_system_prompt(taxonomy_prompt); modules
    without one bake a flat _SYSTEM_PROMPT constant. Modules whose taxonomy JSON
    does not exist yet (the rejection reason module before section 7) are skipped.
    """
    out = {}
    for module in ALL_MODULES:
        name = f'generated.modules.{module}'
        mod = importlib.reload(sys.modules[name]) if name in sys.modules \
            else importlib.import_module(name)
        if module in MODULE_PREFIXES:
            tax_path = TAXONOMY_DIR / f'{module}.json'
            if not tax_path.exists():
                print(f'  {module}: no taxonomy JSON yet - skipped')
                continue
            tax = json.loads(tax_path.read_text(encoding='utf-8'))
            out[module] = mod._build_system_prompt(
                taxonomy_to_rich_prompt(tax, MODULE_PREFIXES[module]))
        else:
            out[module] = mod._SYSTEM_PROMPT
    return out


prompts = build_prompts()

# ---- Verbatim guidance check -------------------------------------------
# Every rule text and every template scenario must survive the round trip
# through Excel, the taxonomy JSON and the prompt builder.
checked = 0
_rule_prompt = prompts[MOD_RULE]
for _rule in RULES:
    assert _rule['rule'] in _rule_prompt, (
        f'rule {_rule["n"]} text missing from the prompt: {_rule["rule"][:70]!r}')
    assert _rule['scenario'] in _rule_prompt, (
        f'rule {_rule["n"]} template scenario missing: {_rule["scenario"][:70]!r}')
    checked += 2
assert RULE_PRECEDENCE in _rule_prompt, 'authored precedence brief missing'
for _n, _cue in RULE_NEGATIVE_CUES.items():
    assert _cue in _rule_prompt, f'negative cue for rule {_n} missing from the prompt'
    checked += 1
assert NA_RULE_DESC in _rule_prompt, 'authored NA description missing'
assert NA_RULE_CODE in _rule_prompt, 'the NA code is not named in the prompt'

# ...and the ground truth must NOT be there.
for _rule in RULES:
    assert _rule['primary_raw'].casefold() not in _rule_prompt.casefold(), (
        f'resolution code {_rule["primary_raw"]!r} leaked into the prompt')

print(f'\nVerbatim check: all {checked} rule texts and template scenarios are in the '
      f'prompt, and no resolution code is.')

# ---- Per-module fingerprints -------------------------------------------
# Stored beside each checkpoint so a prompt edit invalidates only the affected
# module. A single global hash would force a full re-run for a one-module fix.
_PREPROC = (f'{SHORT_CHAR_THRESHOLD}|{STUB_PHRASE}|{TRUNC_CAP}|{TRUNC_HEAD}|'
            f'{TRUNC_TAIL}|{TARGET_YEAR}')


def fingerprint_prompts(prompt_map: dict) -> dict:
    """Hash each prompt; fold the preprocessing parameters into the tagging modules.

    Both tagging modules read the same cut text, and the checkpoint keys on record
    content - so changing a truncation parameter must invalidate them too.
    """
    fps = {}
    for module, text in prompt_map.items():
        salt = _PREPROC if module in TAGGING_MODULES else ''
        fps[module] = hashlib.sha256((text + salt).encode('utf-8')).hexdigest()[:16]
    return fps


PROMPT_FINGERPRINTS = fingerprint_prompts(prompts)

print(f'\n{"module":<26} {"chars":>7} {"~tokens":>8}  fingerprint')
for _m, _p in prompts.items():
    print(f'{_m:<26} {len(_p):>7,} {len(_p) // 4:>8,}  {PROMPT_FINGERPRINTS[_m]}')

In [ ]:
# The full validation-rule prompt, exactly as the model will receive it.
# Long, but this is the one chance to read it before spending money.
print(prompts[MOD_RULE])

## 6. Gate 2 — preprocess the tickets *(free)*

Three things happen here, and each one is a judgement call worth stating.

**1. The stub rule.** 5,844 tickets contain only *"Thank you for your time earlier. The transcript
of our chat is included here for your reference."* — the chat itself is not in the file. A further
band of tickets is under 300 payload characters. Neither can be classified by any model, so they
are excluded from the LLM run and carried into the final output labelled as unclassifiable rather
than silently dropped.

Notebook 07 used a 150-character threshold; on this corpus that catches 94 rows. The distribution
here is different — the 1st percentile is 273 characters — so the threshold is **300**.

**The exclusion is not random**, and the cell prints the proof: `Invalid- advice request` is
heavily over-represented among the short tickets. Any coverage figure in Section 18 has to be read
against that.

**2. Truncation.** Only ~196 conversations exceed 16,000 characters. They are cut head+tail rather
than head-only because the agent's reason for refusing sits at the *end* of the thread — hence the
4,000/11,900 split rather than an even one.

**3. Blindness.** The LLM input is written from an explicit two-column allowlist. `df.drop(...)`
would be a bug waiting for a schema change. Three assertions follow: no column mentions
`resolution`, the header line on disk contains exactly the two allowed names, and no conversation
body contains the literal `Invalid-` (the internal OSVC codes should never appear in customer
correspondence, but if they ever did, the conformance measurement would be worthless).

In [ ]:
work = corpus.copy()
work['conversation_chars_original'] = work[TEXT_COL].astype(str).str.len()
work['payload_chars'] = work[TEXT_COL].map(payload_len)

is_stub  = work[TEXT_COL].str.contains(STUB_PHRASE, case=False, na=False)
is_short = work['payload_chars'] < SHORT_CHAR_THRESHOLD

# ---- Ground-truth leakage ------------------------------------------------
# A handful of transcripts quote an OSVC resolution code verbatim - an agent
# explaining that the team will reject this as Invalid-Captured correct, or an
# internal triage spreadsheet pasted into the thread. Those tickets hand the model
# the answer, so they are excluded rather than scored.
#
# Match the ACTUAL 15-code vocabulary, not a bare Invalid- prefix: the loose
# pattern also fires on unrelated spreadsheet headers (Invalid-Valid Scopus link,
# Invalid-Citation type not valid) that reveal nothing.
_code_tails = sorted({norm_res_code(v).split('-', 1)[1]
                      for v in work[RC_COL].dropna().unique()}, key=len, reverse=True)
CODE_LEAK_RE = (r'invalid\s*-\s*(?:'
                + '|'.join(re.escape(t).replace(r'\ ', r'\s+') for t in _code_tails) + r')')
is_leak = work[TEXT_COL].str.contains(CODE_LEAK_RE, case=False, na=False, regex=True)

# Informational: the looser pattern, so the near-misses are visible too.
_loose = int(work[TEXT_COL].str.contains(r'invalid\s*-', case=False, na=False, regex=True).sum())

excluded_mask = is_stub | is_short | is_leak
work['exclusion_reason'] = np.where(
    is_leak, 'resolution code quoted in the transcript',
    np.where(is_stub, 'chat transcript not captured',
             np.where(is_short, f'payload under {SHORT_CHAR_THRESHOLD} chars', '')))

excluded = work[excluded_mask].copy()
to_classify = work[~excluded_mask].copy().reset_index(drop=True)

print(f'{len(work):,} tickets')
print(f'  chat-transcript stubs                    : {int(is_stub.sum()):,}')
print(f'  payload < {SHORT_CHAR_THRESHOLD} chars                       : {int(is_short.sum()):,}')
print(f'  resolution code quoted in the transcript : {int(is_leak.sum()):,}'
      f'   (of {_loose:,} containing the bare string "Invalid-")')
print(f'  excluded (union)                         : {len(excluded):,}')
print(f'  -> classifiable                          : {len(to_classify):,}')
assert len(excluded) + len(to_classify) == len(work)

# A systemic leak - the code appended to every closed ticket, say - would make the
# whole conformance measurement worthless. A handful is a data quirk.
_leak_rate = int(is_leak.sum()) / len(work)
assert _leak_rate < 0.001, (
    f'{_leak_rate:.2%} of tickets quote a resolution code verbatim. That is systemic, not '
    f'incidental: the model would be reading the answer off the transcript. Investigate '
    f'before continuing.')
if int(is_leak.sum()):
    print('\nExcluded for quoting a resolution code:')
    for _, _r in work[is_leak].head(10).iterrows():
        _m = re.search(CODE_LEAK_RE, _r[TEXT_COL], flags=re.IGNORECASE)
        _ctx = _r[TEXT_COL][max(0, _m.start() - 70):_m.end() + 25].replace('\n', ' ')
        print(f'  {_r[ID_COL]}  ...{_ctx}...')

# Exclusion bias: if one resolution code dominates the excluded set, every coverage
# figure downstream is computed on a skewed denominator. Show it rather than hide it.
_ex = excluded[RC_COL].value_counts()
_all = work[RC_COL].value_counts()
_bias = pd.DataFrame({'excluded': _ex, 'total': _all})
_bias['share_excluded'] = (_bias['excluded'] / _bias['total']).map(lambda v: f'{v:.1%}')
print('\nExclusion by resolution code (NOT random - read this):')
print(_bias.sort_values('excluded', ascending=False).to_string())
excluded.to_csv(EXCLUDED_CSV, index=False)
print(f'\nWrote {EXCLUDED_CSV.relative_to(BASE)}')

# ---- Truncate long conversations ---------------------------------------
_was_long = to_classify['conversation_chars_original'] > TRUNC_CAP
to_classify['was_truncated'] = _was_long
to_classify[TEXT_COL] = to_classify[TEXT_COL].map(truncate_middle)
_kept = to_classify[TEXT_COL].str.len().sum()
_orig = to_classify['conversation_chars_original'].sum()
print(f'\nTruncated {int(_was_long.sum())} of {len(to_classify):,} conversations to '
      f'{TRUNC_CAP:,} chars (head {TRUNC_HEAD:,} + tail {TRUNC_TAIL:,})')
print(f'  text retained: {_kept:,} of {_orig:,} chars ({_kept / _orig:.1%})')
assert to_classify[TEXT_COL].str.len().max() <= TRUNC_CAP + 64


In [ ]:
# ---- Write the LLM input, blind to the answer --------------------------
# Explicit allowlist. Never df.drop(...): that leaks on a schema change.
LLM_COLS = [ID_COL, TEXT_COL]
llm_input = to_classify[LLM_COLS].copy()
llm_input.to_csv(STAGE_INPUT, index=False)

# 1. no column name mentions the ground truth
_leaky_cols = [c for c in llm_input.columns if 'resolution' in c.lower() or 'rc_' in c.lower()]
assert not _leaky_cols, f'ground-truth columns in the LLM input: {_leaky_cols}'
assert list(llm_input.columns) == LLM_COLS, llm_input.columns

# 2. the header line actually written to disk
with open(STAGE_INPUT, encoding='utf-8') as _f:
    _header = _f.readline().strip()
assert _header == ','.join(LLM_COLS), f'unexpected header on disk: {_header!r}'

# 3. no resolution code from the 15-code vocabulary survives in any body
_still = llm_input[TEXT_COL].str.contains(CODE_LEAK_RE, case=False, na=False, regex=True)
assert not _still.any(), (
    f'{int(_still.sum())} conversation(s) still quote a resolution code after the exclusion '
    f'in the previous cell: {llm_input.loc[_still, ID_COL].head(5).tolist()}')

print(f'Wrote {STAGE_INPUT.relative_to(BASE)}')
print(f'  {len(llm_input):,} rows x {len(llm_input.columns)} cols: {LLM_COLS}')
print(f'  header on disk: {_header}')
print('\nBlindness checks passed: the model sees the ticket id and the conversation, '
      'nothing else.')
print(f'\nMean conversation length: {llm_input[TEXT_COL].str.len().mean():,.0f} chars '
      f'(~{llm_input[TEXT_COL].str.len().mean() / 4:,.0f} tokens)')
print(f'Estimated cost at $0.0065/record/module x {len(TAGGING_MODULES)} modules:')
for _n in (SMOKE_N, PROBE_N, SAMPLE_N, len(llm_input)):
    print(f'  {_n:>7,} tickets -> ${_n * 0.0065 * len(TAGGING_MODULES):>8,.2f}')


## 7. Gate 3 — discover the rejection-reason taxonomy *(~$3)*

**This is the first cell that spends money.**

The validation rules are top-down: someone wrote them from the coverage guidelines. The rejection
reasons are bottom-up — extracted from what agents actually told customers. The two should agree
where the matrix is complete, and diverge where it is not. That divergence is the finding.

`rca build discover` runs two passes:

- **Pass 1** sends batches of transcripts to the model with the `extract_prompt` from the modules
  sheet and collects one abstract reason per ticket, then collapses near-duplicates at 0.75 token
  overlap. The raw counter is saved as `csd_rejection_reason_extracted.json`.
- **Pass 2** makes a *single* call over the top-N items and returns a two-level taxonomy. Being a
  single call, it is the fragile step: malformed JSON yields an empty draft. Section 8 checks for
  that, and because Pass 1's output is preserved, only Pass 2 needs re-running if it fails.

**One wrinkle on Windows.** The `discover` subprocess finishes its work, writes all three draft
artefacts, prints `DISCOVERY COMPLETE` — and then segfaults during interpreter shutdown
(`0xC0000005`, exit code 3221225477). It is a teardown crash in the async HTTP stack, not a
failed run; `rca run` is unaffected, so this is the only cell that has to cope with it.
The cell below tolerates that one exit code, and only when the draft file was actually rewritten
and `DISCOVERY COMPLETE` appeared in the output. Every other failure still raises.

The sample is stratified and written **without** the resolution code, for the same reason the main
input is. `discover` samples internally with `df.sample(random_state=42)`, so handing it a file
that is already exactly `DISCOVERY_N` rows makes its own sampling a no-op and keeps the
stratification we chose. A plain random 400 out of 50,721 would contain roughly four
`Invalid- book` tickets and none at all from `Invalid- Out of coverage`.

In [ ]:
disc_sample = stratified_sample(to_classify, RC_COL, DISCOVERY_N,
                                min_per=DISCOVERY_MIN, max_per=DISCOVERY_MAX)
disc_sample[LLM_COLS].to_csv(DISCOVERY_CSV, index=False)

print(f'Discovery sample: {len(disc_sample)} tickets '
      f'(floor {DISCOVERY_MIN}, ceiling {DISCOVERY_MAX} per resolution code)')
print(disc_sample[RC_COL].value_counts().to_string())
assert disc_sample[RC_COL].nunique() == to_classify[RC_COL].nunique(), (
    'a resolution code is missing from the discovery sample')

with open(DISCOVERY_CSV, encoding='utf-8') as _f:
    assert _f.readline().strip() == ','.join(LLM_COLS)
print(f'\nWrote {DISCOVERY_CSV.relative_to(BASE)} (id + conversation only)')

In [ ]:
# ~$3, ~4 minutes. Pass 1 over the sample plus one Pass 2 normalisation call.
#
# KNOWN WINDOWS ISSUE: rca build discover completes all its work, writes
# all three draft artefacts, prints DISCOVERY COMPLETE - and then the interpreter
# segfaults during shutdown with 0xC0000005 (exit code 3221225477, or -1073741819
# as PowerShell reports it). It is a teardown crash in the async HTTP stack, not a
# failure of the discovery itself. rca run does not have it, so this is the
# only cell that needs to tolerate it.
#
# Tolerating it is narrow and evidence-based: ONLY that exact exit code, and only
# when the draft file was actually written after the command started. Any other
# non-zero exit, or a missing/stale draft, still raises.
_SHUTDOWN_CRASH = (3221225477, -1073741819)   # 0xC0000005, both signednesses


def run_discovery(module: str, data_csv: Path, *, sample_n: int, top_n: int = 80) -> Path:
    """Run the discovery CLI and return the draft path, tolerating the shutdown crash."""
    draft = DRAFT_DIR / f'{module}_draft.json'
    before = draft.stat().st_mtime if draft.exists() else -1.0

    cmd = [sys.executable, '-m', 'rca_pipeline.cli', 'discover',
           '--input', str(DEFN_XLSX), '--data', str(data_csv), '--modules', module,
           '--sample-n', str(sample_n), '--top-n', str(top_n),
           '--id-col', ID_COL, '--text-col', TEXT_COL]
    print('$ ' + ' '.join(str(c) for c in cmd), flush=True)
    r = subprocess.run(cmd, text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
    print(r.stdout or '', end='')
    print(r.stderr or '', end='')

    wrote = draft.exists() and draft.stat().st_mtime > before
    completed = 'DISCOVERY COMPLETE' in (r.stdout or '') + (r.stderr or '')

    if r.returncode == 0:
        pass
    elif r.returncode in _SHUTDOWN_CRASH and wrote and completed:
        print(f'\nNOTE: the process exited {r.returncode} (0xC0000005) AFTER finishing. '
              f'{draft.name} was written and DISCOVERY COMPLETE was printed, so the crash '
              f'is in interpreter teardown and the result is sound. Continuing.')
    else:
        raise RuntimeError(
            f'discover failed (exit {r.returncode}); draft written: {wrote}, '
            f'DISCOVERY COMPLETE printed: {completed}. This is NOT the known teardown '
            f'crash - read the output above.')

    assert wrote, f'{draft} was not written by this run'
    return draft


run_discovery(MOD_REASON, DISCOVERY_CSV, sample_n=DISCOVERY_N)


## 8. Accept the discovered taxonomy

The draft is accepted automatically — no hard stop — but not blindly. This cell prints every
discovered leaf next to its Pass 1 frequency so the taxonomy is readable in the output, then
enforces three things:

1. **Non-empty and sane size.** A collapsed Pass 2 produces `{}`; an over-eager one produces a
   hundred near-duplicates. Both are caught.
2. **The two structurally required leaves exist.** This corpus contains 8,070 status enquiries and
   ~6,200 tickets with no substantive agent reply. If discovery did not surface a home for either,
   the model has nowhere to put them and will spread them across unrelated reasons. Rather than
   fail the run, the cell **appends an authored leaf** and says so loudly.
3. **No leaf is a request type.** "Missing article" describes what the customer asked for, not why
   it was refused; such leaves are reported for review but left in place.

### Overriding the draft

`REASON_TAXONOMY_OVERRIDE` at the top of the cell is `None`, so the draft wins. To substitute a
hand-edited taxonomy later, paste a list of `(l1_label, l1_desc, leaf_label, leaf_desc)` tuples
there and re-run from this cell. Section 9 rebuilds, the fingerprints change, the checkpoint guard
discards the old results, and only the tagging pass is repeated — discovery is not.

In [ ]:
REASON_TAXONOMY_OVERRIDE = None   # paste a list of 4-tuples here to override the draft

_draft_path = DRAFT_DIR / f'{MOD_REASON}_draft.json'
_extract_path = DRAFT_DIR / f'{MOD_REASON}_extracted.json'
assert _draft_path.exists(), (
    f'{_draft_path} not found - section 7 did not produce a draft. '
    f'Check the discover output above for a Pass 2 JSON error.')

_draft = json.loads(_draft_path.read_text(encoding='utf-8'))

# ---- Pass 1 evidence ----------------------------------------------------
# discovery.py writes {"domain":..., "mode":..., "items":[{"item":..., "frequency":...}]}
# The item wording is pre-normalisation, so it does not match the Pass 2 leaf labels
# directly - each item is attached to its best-matching leaf by token overlap below.
PASS1_ITEMS = []
if _extract_path.exists():
    _raw = json.loads(_extract_path.read_text(encoding='utf-8'))
    PASS1_ITEMS = [(norm_label(d.get('item', '')), int(d.get('frequency', 0)))
                   for d in _raw.get('items', []) if norm_label(d.get('item', ''))]
print(f'Pass 1 extracted {len(PASS1_ITEMS)} distinct items '
      f'covering {sum(f for _, f in PASS1_ITEMS)} ticket mentions.\n')
for _item, _f in PASS1_ITEMS[:40]:
    print(f'  {_f:>4}  {_item}')
if len(PASS1_ITEMS) > 40:
    print(f'  ... and {len(PASS1_ITEMS) - 40} more')


def draft_to_tuples(draft: dict) -> list:
    """Flatten a discovered taxonomy dict into (l1_label, l1_desc, leaf_label, leaf_desc)."""
    out = []
    for _l1code, node in draft.items():
        if not isinstance(node, dict):
            continue
        l1_label = norm_label(node.get('label', _l1code))
        l1_desc = norm_prose(node.get('description', ''))
        for _lc, leaf in node.get('children', {}).items():
            if not isinstance(leaf, dict):
                continue
            out.append((l1_label, l1_desc,
                        norm_label(leaf.get('label', _lc)),
                        norm_prose(leaf.get('description', ''))))
    return out


def _tokens(text: str) -> set:
    return {w for w in re.findall(r'[a-z]{4,}', text.lower())}


def leaf_frequencies(taxonomy: list, items: list) -> dict:
    """Attach each Pass 1 item to its single best-matching leaf, and sum frequencies.

    Overlap is Jaccard over 4+ character words. An item that matches nothing is
    reported separately rather than silently dropped - a large unattached total
    means Pass 2 invented labels that do not describe what Pass 1 actually found.
    """
    leaf_tokens = {t[2]: _tokens(t[2] + ' ' + t[3]) for t in taxonomy}
    freq = {t[2]: 0 for t in taxonomy}
    unattached = []
    for item, f in items:
        it = _tokens(item)
        if not it:
            continue
        best, best_score = None, 0.0
        for label, lt in leaf_tokens.items():
            if not lt:
                continue
            score = len(it & lt) / len(it | lt)
            if score > best_score:
                best, best_score = label, score
        if best is not None and best_score >= 0.15:
            freq[best] += f
        else:
            unattached.append((item, f))
    return freq, unattached


DISCOVERED = draft_to_tuples(_draft)
_freq, _unattached = leaf_frequencies(DISCOVERED, PASS1_ITEMS)

print(f'\n\nDraft: {len({t[0] for t in DISCOVERED})} L1 groups, {len(DISCOVERED)} leaves')
print('(counts are Pass 1 mentions attached to each leaf by wording overlap)\n')
_last_l1 = None
for _l1, _l1d, _leaf, _leafd in DISCOVERED:
    if _l1 != _last_l1:
        print(f'\n  {_l1}')
        _last_l1 = _l1
    print(f'    [{_freq.get(_leaf, 0):>4}] {_leaf}')
    if _leafd:
        print(f'           {_leafd[:150]}')

if _unattached:
    _n = sum(f for _, f in _unattached)
    print(f'\n{len(_unattached)} Pass 1 item(s) ({_n} mentions, '
          f'{_n / max(1, sum(f for _, f in PASS1_ITEMS)):.0%}) matched no leaf:')
    for _item, _f in sorted(_unattached, key=lambda x: -x[1])[:10]:
        print(f'  {_f:>4}  {_item}')
    print('  A large share here means Pass 2 named groups that do not describe what Pass 1 '
          'found.')


In [ ]:
# ---- Guardrails --------------------------------------------------------
REQUIRED_LEAVES = [
    # 'no actionable request' is deliberately NOT in this list. The first real
    # discovery run produced a leaf called 'Non-Actionable Requests' whose
    # description was about requests that are unclear or lack information - a
    # different thing from a coverage-status enquiry. Matching on that phrase let
    # the check pass while the corpus's ~8,000 status enquiries still had no home,
    # which is precisely the gap section 16 exists to surface. Require the
    # status/coverage concept itself.
    {'name': 'status enquiry',
     'match': lambda t: any(w in (t[2] + ' ' + t[3]).lower()
                            for w in ('status enquir', 'status inquir', 'status query',
                                      'coverage enquir', 'coverage inquir',
                                      'coverage query', 'asked about the status',
                                      'current status', 'status of the source',
                                      'status of a source', 'status of the journal')),
     'l1': 'Not an actionable request',
     'l1_desc': 'The ticket did not contain a request that could be actioned.',
     'leaf': 'Status or coverage enquiry only - no actionable request',
     'leaf_desc': (
         'The customer asked about the current status or coverage of a source or an article '
         'rather than asking for content to be added or corrected. The agent answered the '
         'question; there was no request to action, so nothing was refused on content grounds. '
         'Typical wording asks whether a journal is indexed, when it was discontinued, or what '
         'the current evaluation status is.')},
    {'name': 'no reason stated',
     'match': lambda t: any(w in (t[2] + ' ' + t[3]).lower()
                            for w in ('no reason', 'not stated', 'no substantive',
                                      'unclear', 'cannot be determined', 'insufficient')),
     'l1': 'Not an actionable request',
     'l1_desc': 'The ticket did not contain a request that could be actioned.',
     'leaf': 'No reason stated in the ticket',
     'leaf_desc': (
         'The agent gave no substantive ground for refusing the request. The ticket is a chat '
         'transcript stub, a handoff to another team, an acknowledgement, or boilerplate only. '
         'Use this rather than inferring a reason that the ticket does not state.')},
]

REASON_TAXONOMY = list(REASON_TAXONOMY_OVERRIDE or DISCOVERED)
_appended = []
for _req in REQUIRED_LEAVES:
    if any(_req['match'](t) for t in REASON_TAXONOMY):
        print(f'OK   required leaf present: {_req["name"]}')
    else:
        REASON_TAXONOMY.append((_req['l1'], _req['l1_desc'], _req['leaf'], _req['leaf_desc']))
        _appended.append(_req['name'])
        print(f'ADD  required leaf missing, appended an authored one: {_req["name"]}')

if _appended:
    print(f'\n*** Discovery did not surface {len(_appended)} structurally required leaf/leaves '
          f'({", ".join(_appended)}).\n'
          f'    They were appended rather than failing the run, because this corpus contains '
          f'thousands of\n    tickets of each kind and the model needs somewhere to put them. '
          f'Worth reading the draft above\n    to understand what Pass 2 merged them into.')

# Leaves that name a request rather than a ground for refusal - report, do not remove.
_REQUEST_WORDS = ('missing article', 'missing document', 'add article', 'correction request',
                  'customer request', 'indexing request')
_suspect = [t[2] for t in REASON_TAXONOMY
            if any(w in t[2].lower() for w in _REQUEST_WORDS)]
if _suspect:
    print(f'\nReview: {len(_suspect)} leaf/leaves name a REQUEST rather than a reason for '
          f'refusing it.\n  ' + '\n  '.join(_suspect))

# ---- Shape ------------------------------------------------------------
_n_l1 = len({t[0] for t in REASON_TAXONOMY})
_n_leaf = len(REASON_TAXONOMY)
_dupes = [k for k, v in Counter(norm_cmp(t[2]) for t in REASON_TAXONOMY).items() if v > 1]
print(f'\nFinal rejection-reason taxonomy: {_n_l1} L1 groups, {_n_leaf} leaves')
assert _n_leaf >= 8, (
    f'only {_n_leaf} leaves - Pass 2 collapsed. The Pass 1 counter is preserved at '
    f'{_extract_path.name}; re-run section 7 to redo Pass 2.')
assert _n_leaf <= 60, f'{_n_leaf} leaves is too many to classify against reliably'
assert not _dupes, f'duplicate leaf labels would make the codes ambiguous: {_dupes}'
assert all(t[3] for t in REASON_TAXONOMY), 'a leaf has no description - the model needs one'
print('Shape checks passed.')

## 9. Rewrite the workbook with the discovered taxonomy and rebuild

Excel stays the single source of truth. This is why the notebook does **not** use
`rca build promote-drafts`: promote copies the draft over the live JSON, and the very next
`build` overwrites it again from the Excel — which at that point is still empty. Writing the
taxonomy back into the workbook and rebuilding is the only version that survives a rebuild.

`discovery_mode` flips from `full` back to `''` here (`build_modules_sheet` keys off whether
`REASON_TAXONOMY` is populated), so the module is a classifier from now on rather than a
discovery target.

The fingerprints recomputed at the end **will differ** from Section 5 — the rejection-reason
prompt now has a taxonomy in it. That is correct, and the checkpoint guard in Section 10 will act
on it.

In [ ]:
modules_df, fields_df, taxonomy_df = write_definition_workbook()
_disc_mode = modules_df.loc[modules_df['module_name'] == MOD_REASON, 'discovery_mode'].iloc[0]
assert _disc_mode == '', (
    f'discovery_mode is still {_disc_mode!r} - the module would route into discovery at '
    f'classification time')

print(f'Rewrote {DEFN_XLSX.relative_to(BASE)}')
for _m in ALL_MODULES:
    _sub = taxonomy_df[taxonomy_df['module_name'] == _m]
    print(f'  {_m:<24} {int((_sub["level"] == "L1").sum()):>2} L1 / '
          f'{int((_sub["level"] == "L2").sum()):>2} L2')

run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
         '--input', str(DEFN_XLSX), '--force-all'], 'build --force-all (pass 2)')

In [ ]:
prompts = build_prompts()
assert set(prompts) == set(ALL_MODULES), (
    f'expected a prompt for every module, got {sorted(prompts)}')

# The rejection-reason prompt must now carry every discovered leaf and its guidance.
_reason_prompt = prompts[MOD_REASON]
for _l1, _l1d, _leaf, _leafd in REASON_TAXONOMY:
    assert _leaf in _reason_prompt, f'leaf missing from the reason prompt: {_leaf!r}'
    assert _leafd in _reason_prompt, f'leaf description missing: {_leafd[:70]!r}'
for _rule in RULES:
    assert _rule['primary_raw'].casefold() not in _reason_prompt.casefold(), (
        f'resolution code {_rule["primary_raw"]!r} leaked into the reason prompt')

REASON_LEAVES = leaf_labels(MOD_REASON)
RULE_LEAVES   = leaf_labels(MOD_RULE)
NO_REASON_CODES = [c for c, lab in REASON_LEAVES.items()
                   if 'no reason' in lab.lower() or 'not stated' in lab.lower()]
print(f'{MOD_RULE}: {len(RULE_LEAVES)} leaf codes')
print(f'{MOD_REASON}: {len(REASON_LEAVES)} leaf codes')
print(f'"no reason stated" code(s): {NO_REASON_CODES}')

_old = dict(PROMPT_FINGERPRINTS)
PROMPT_FINGERPRINTS = fingerprint_prompts(prompts)
print(f'\n{"module":<26} {"chars":>7} {"~tokens":>8}  fingerprint   changed?')
for _m, _p in prompts.items():
    _chg = 'NEW' if _m not in _old else ('yes' if _old[_m] != PROMPT_FINGERPRINTS[_m] else 'no')
    print(f'{_m:<26} {len(_p):>7,} {len(_p) // 4:>8,}  '
          f'{PROMPT_FINGERPRINTS[_m]}  {_chg}')

In [ ]:
# The full rejection-reason prompt, exactly as the model will receive it.
print(prompts[MOD_REASON])

## 10. Profiles and the checkpoint guard

**One profile, both tagging modules.** `rca run` runs the modules listed in a profile
sequentially against one input and writes one checkpoint, keyed per record with a *per-module*
sentinel (`csdvr_code` / `csdrr_code`, confirmed in Section 5). So a single `run_stage()` call
tags both, and if the run dies halfway through the second module the first is not repeated.

**The guard exists because resuming is the default and it is usually wrong after an edit.**
`rca run` resumes from `runs/checkpoints/<input-stem>_checkpoint.jsonl` whenever every
module already has a result for a record — it logs `skip=N todo=0` and makes no LLM calls. That is
exactly right after an interruption and exactly wrong after a taxonomy change. `_sync_checkpoint`
compares the current prompt fingerprint with the one stamped beside the checkpoint and deletes the
checkpoint when they differ. Being per-module, editing the rule prompt does not throw away the
rejection-reason results.

Checkpoints are keyed by the input file's **stem**, not its full path, so two notebooks using the
same basename would silently share one. `run_stage()` asserts every input is named `csd_*`, which
makes a collision with notebook 07's `sd_*` files impossible.

`write_profile` validates its keys against `rca run --help` rather than a hardcoded list —
a key the runner does not accept would otherwise be ignored in silence.

In [ ]:
# ---- Which keys does rca run actually accept? -------------------
# Read the flags off --help rather than regex-scanning the source: it is the same
# list argparse will really use, and it doubles as a check that rca run
# imports cleanly under this interpreter.
_help = subprocess.run([sys.executable, '-m', 'rca_pipeline.cli', 'run', '--help'],
                       text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
assert _help.returncode == 0, f'rca run --help failed:\n{_help.stderr[-2000:]}'
_dests = {f.replace('-', '_') for f in re.findall(r'--([a-z][a-z0-9-]+)', _help.stdout)}
assert {'input', 'output', 'batch_size', 'max_chars', 'modules'} <= _dests, \
    f'could not parse rca run flags (got {sorted(_dests)})'

TUNING = {
    'batch_size': 5, 'max_chars': 40_000, 'concurrency': 4,
    'retries': 5, 'max_tokens': 8192, 'track_costs': True,
    'cost_report_dir': 'runs/cost_reports',
}


def write_profile(path: Path, *, comment, input_rel, output_rel, modules) -> Path:
    spec = {'_comment': comment, 'input': input_rel, 'output': output_rel,
            'taxonomy_dir': 'generated/taxonomies',
            'text_col': TEXT_COL, 'id_col': ID_COL, 'modules': modules, **TUNING}
    unknown = [k for k in spec if k != '_comment' and k not in _dests]
    assert not unknown, f'{path.name}: keys rca run would silently ignore: {unknown}'
    path.write_text(json.dumps(spec, indent=2) + '\n', encoding='utf-8')
    return path


write_profile(
    PROFILE_TAG,
    comment=(f'Tags every CSD {TARGET_YEAR} ticket with a validation rule (from the '
             f'triangulation matrix) and a rejection reason (discovered from the tickets). '
             f'Both modules share this profile so they also share one checkpoint, with a '
             f'per-module sentinel. Input is written by '
             f'notebooks/08_csd_2026_validation_rules.ipynb - stubs removed, conversations '
             f'capped at {TRUNC_CAP} chars, ticket id and conversation only - not the raw '
             f'ticket file.'),
    input_rel=SAMPLE_CSV.relative_to(BASE).as_posix(),
    output_rel=MAIN_OUT.relative_to(BASE).as_posix(),
    modules=TAGGING_MODULES)

write_profile(
    PROFILE_PROP,
    comment=('Gap pass. Runs only on tickets the rule module tagged as having no applicable '
             'validation rule, and drafts the rule that would cover them. No taxonomy: the '
             'output is free text for CSD review, not a classification.'),
    input_rel=PROPOSAL_CSV.relative_to(BASE).as_posix(),
    output_rel=PROPOSAL_OUT.relative_to(BASE).as_posix(),
    modules=[MOD_PROPOSAL])

print(f'Wrote {PROFILE_TAG.relative_to(BASE)}')
print(f'      {PROFILE_PROP.relative_to(BASE)}')
print(f'\nTuning: {TUNING}')
print(f'Modules per tagging call: {TAGGING_MODULES}')

In [ ]:
def checkpoint_files(input_path) -> list:
    """The checkpoint artefacts rca run keeps for a given input file."""
    stem = Path(input_path).stem
    return [CHECKPOINT_DIR / f'{stem}_checkpoint.jsonl',
            CHECKPOINT_DIR / f'{stem}_bad_uids.txt']


def _fingerprint_file(input_path) -> Path:
    return CHECKPOINT_DIR / f'{Path(input_path).stem}_checkpoint.fingerprint'


def _read_fingerprints(fp: Path) -> dict:
    """Read a {module: hash} fingerprint file, tolerating older formats."""
    if not fp.exists():
        return {}
    try:
        stored = json.loads(fp.read_text(encoding='utf-8'))
    except (json.JSONDecodeError, UnicodeDecodeError):
        return {}
    return stored if isinstance(stored, dict) else {}


def _sync_checkpoint(input_path, modules) -> None:
    """Drop this input's checkpoint if any of these modules' prompts have changed.

    The checkpoint is one file shared by every module running against this input,
    so it cannot be invalidated per module - but the fingerprint file is keyed per
    module, so we can tell WHICH prompt changed and say so.
    """
    fp = _fingerprint_file(input_path)
    stored = _read_fingerprints(fp)
    stale = [m for m in modules
             if PROMPT_FINGERPRINTS.get(m) is not None
             and stored.get(m) != PROMPT_FINGERPRINTS[m]]
    if not stale:
        return
    dropped = [p.name for p in checkpoint_files(input_path) if p.exists()]
    for p in checkpoint_files(input_path):
        p.unlink(missing_ok=True)
    if dropped:
        for m in stale:
            print(f'  {m}: prompt changed since that checkpoint '
                  f'({stored.get(m) or "no fingerprint"} -> {PROMPT_FINGERPRINTS[m]})')
        print(f'  discarded {", ".join(dropped)}')


def _stamp_checkpoint(input_path, modules) -> None:
    fp = _fingerprint_file(input_path)
    fp.parent.mkdir(parents=True, exist_ok=True)
    stored = _read_fingerprints(fp)
    for m in modules:
        if PROMPT_FINGERPRINTS.get(m) is not None:
            stored[m] = PROMPT_FINGERPRINTS[m]
    fp.write_text(json.dumps(stored, indent=2), encoding='utf-8')


def run_stage(profile: Path, *, modules, input_path=None, output_path=None,
              label='', extra_args=()) -> Path:
    """Run one pipeline stage as a subprocess. Returns the output path.

    CLI flags override profile values (rca run applies a profile key only
    when the matching arg is still at its default), so input/output can be
    redirected for a smoke test without editing the profile.
    """
    spec = json.loads(profile.read_text(encoding='utf-8'))
    resolved_in = Path(input_path) if input_path else BASE / spec['input']
    resolved_out = Path(output_path) if output_path else BASE / spec['output']
    resolved_out.parent.mkdir(parents=True, exist_ok=True)

    # Checkpoints are keyed by the input file's STEM, not its full path, so two
    # notebooks using the same basename share - and silently resume from - each
    # other's checkpoint. Namespacing every stage input csd_* makes that impossible.
    assert resolved_in.stem.startswith('csd_'), (
        f'{resolved_in.name}: stage inputs must be named csd_* so their checkpoint '
        f'stem cannot collide with another notebook')
    assert resolved_in.exists(), f'stage input does not exist: {resolved_in}'

    cmd = [sys.executable, '-m', 'rca_pipeline.cli', 'run', '--profile', str(profile),
           '--input', str(resolved_in), '--output', str(resolved_out), *map(str, extra_args)]

    banner = label or profile.stem
    print('=' * 78)
    print(f'STAGE: {banner}')
    print('=' * 78)
    _sync_checkpoint(resolved_in, modules)
    print(' '.join(str(c) for c in cmd), flush=True)

    err_lines = []

    def _drain(stream):
        for line in iter(stream.readline, ''):
            err_lines.append(line)

    t0 = time.time()
    proc = subprocess.Popen(cmd, text=True, encoding='utf-8', errors='replace',
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                            bufsize=1, env=SUBPROCESS_ENV)
    t = threading.Thread(target=_drain, args=(proc.stderr,), daemon=True)
    t.start()
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.wait()
    t.join(timeout=5)

    if proc.returncode != 0:
        print(f'\nExit code {proc.returncode}')
        if err_lines:
            print('--- stderr (last 60 lines) ---')
            print(''.join(err_lines[-60:]))
        raise RuntimeError(f'{banner} failed with exit code {proc.returncode}')

    _stamp_checkpoint(resolved_in, modules)
    print(f'\n-> {banner} done in {time.time() - t0:.0f}s  ->  {resolved_out}')
    return resolved_out


print('Checkpoint guard armed. Fingerprints:')
for _m in ALL_MODULES:
    print(f'  {_m:<26} {PROMPT_FINGERPRINTS[_m]}')

## 11. Gate 4 — 40-ticket smoke test *(~$0.60)*

Cheap enough to run after any prompt edit. It answers three questions the full run cannot answer
cheaply: does the pipeline produce output at all, are the codes valid taxonomy leaves, and is the
evidence field a real quote rather than a paraphrase.

The last one matters more than it looks. If `csdvr_evidence` does not appear in the transcript,
the model is reconstructing rather than reading, and the justification alongside it cannot be
trusted either. `utils._evidence_in_transcript` is the same fuzzy check the pipeline itself uses.

Read the five printed examples. That is the point of the gate.

In [ ]:
smoke = stratified_sample(to_classify, RC_COL, SMOKE_N, min_per=2, seed=RANDOM_SEED)
smoke[LLM_COLS].to_csv(SMOKE_CSV, index=False)
print(f'Smoke set: {len(smoke)} tickets across {smoke[RC_COL].nunique()} resolution codes')

smoke_out = run_stage(PROFILE_TAG, modules=TAGGING_MODULES,
                      input_path=SMOKE_CSV,
                      output_path=SMOKE_DIR / 'csd_smoke_output.xlsx',
                      label=f'GATE 4 - smoke, {len(smoke)} tickets, both modules')

In [ ]:
sm = read_stage_output(smoke_out)
print(f'{len(sm)} rows x {len(sm.columns)} cols')
print('columns:', [c for c in sm.columns if c.startswith(('csdvr', 'csdrr'))])
assert len(sm) == len(smoke), f'expected {len(smoke)} rows back, got {len(sm)}'

RULE_LEAVES = leaf_labels(MOD_RULE)
REASON_LEAVES = leaf_labels(MOD_REASON)
for _col, _valid, _name in ((f'{STUB_RULE}_code', RULE_LEAVES, 'validation rule'),
                            (f'{STUB_REASON}_code', REASON_LEAVES, 'rejection reason')):
    _blank = sm[_col].isna() | (sm[_col].astype(str).str.strip() == '')
    _bad = ~_blank & ~sm[_col].astype(str).str.strip().isin(_valid)
    print(f'\n{_name}: {int(_blank.sum())} blank, {int(_bad.sum())} not a taxonomy leaf')
    if _bad.any():
        print(sm.loc[_bad, [ID_COL, _col]].to_string(index=False))
    assert not _blank.any(), f'{int(_blank.sum())} blank {_name} codes on a 40-ticket run'
    assert not _bad.any(), f'{int(_bad.sum())} {_name} codes are not taxonomy leaves'

print('\nValidation rule distribution:')
print(sm[f'{STUB_RULE}_label'].value_counts().to_string())
print('\nRejection reason distribution:')
print(sm[f'{STUB_REASON}_label'].value_counts().to_string())

# ---- Is the evidence a real quote? -------------------------------------
from rca_pipeline.utils import _evidence_in_transcript

_texts = dict(zip(smoke[ID_COL], smoke[TEXT_COL]))
for _stub in (STUB_RULE, STUB_REASON):
    _col = f'{_stub}_evidence'
    _rows = sm[sm[_col].astype(str).str.strip() != '']
    _hits = sum(_evidence_in_transcript(str(r[_col]), _texts.get(r[ID_COL], ''))
                for _, r in _rows.iterrows())
    _rate = _hits / max(1, len(_rows))
    print(f'\n{_col}: {len(_rows)}/{len(sm)} non-empty, {_hits} found in the transcript '
          f'({_rate:.0%})')
    assert _rate >= 0.80, (
        f'only {_rate:.0%} of {_col} quotes are actually in the transcript - the model is '
        f'paraphrasing, so its justifications cannot be trusted either')

In [ ]:
# Read these. Is the justification reasoning, or is it restating the label?
for _, r in sm.head(5).iterrows():
    print('=' * 78)
    print(f'{r[ID_COL]}')
    print(f'  RULE   {r[f"{STUB_RULE}_code"]:<10} {r[f"{STUB_RULE}_label"]}')
    print(f'         applies={r[f"{STUB_RULE}_applies"]!r}')
    print(f'         why: {r[f"{STUB_RULE}_justification"]}')
    print(f'         ev : {str(r[f"{STUB_RULE}_evidence"])[:200]}')
    print(f'  REASON {r[f"{STUB_REASON}_code"]:<10} {r[f"{STUB_REASON}_label"]}')
    print(f'         why: {r[f"{STUB_REASON}_justification"]}')
    print(f'         ev : {str(r[f"{STUB_REASON}_evidence"])[:200]}')
    _truth = smoke.loc[smoke[ID_COL] == r[ID_COL], RC_COL].iloc[0]
    print(f'  (agent actually logged: {_truth})')

## 12. Gate 5 — 1,000-ticket conformance probe *(~$14)*

The first real measurement, at 8% of the main run's cost. Three things get checked, and any of
them failing is worth stopping for:

1. **Blank rate ≤ 0.5%.** At 40 tickets a single blank is invisible; at 1,000 it is a rate.
2. **`csdvr_applies` agrees with `csdvr_code == CSDVR4.1` at least 98% of the time.** The field is
   deliberately redundant. Disagreement means the model is not reading its own taxonomy — it is
   answering "does a rule apply" and picking a code as two unrelated decisions.
3. **Conformance among mappable tickets ≥ 40%.** Below that, the prompt is broken rather than the
   data being hard, and it is worth finding out before spending $156.

Note which bound goes with which rate. `conformance among mappable` is the headline and its
maximum is **100%** — the 78% from Section 2 is the matrix's code coverage and bounds
`conformance_strict` instead. Section 18.3 prints the pairing explicitly.

In [ ]:
probe = stratified_sample(to_classify, RC_COL, PROBE_N, min_per=25, seed=RANDOM_SEED + 1)
probe[LLM_COLS].to_csv(PROBE_CSV, index=False)
print(f'Probe set: {len(probe)} tickets')
print(probe[RC_COL].value_counts().to_string())

_t0 = time.time()
probe_out = run_stage(PROFILE_TAG, modules=TAGGING_MODULES,
                      input_path=PROBE_CSV,
                      output_path=SMOKE_DIR / 'csd_probe_output.xlsx',
                      label=f'GATE 5 - conformance probe, {len(probe)} tickets')
print(f'\nThroughput: {len(probe) / (time.time() - _t0):.2f} tickets/s')

In [ ]:
def derive_conformance(tagged: pd.DataFrame, truth: pd.DataFrame) -> pd.DataFrame:
    """Join a tagged output to the ground truth and derive the conformance columns.

    NOTE the column naming: nothing may start with `resolution_`, which is in
    utils._STRUCTURAL_PREFIXES and would be silently stripped if this frame were
    ever fed back into the pipeline. Hence rc_* and conf_*.
    """
    rule_code_col = f'{STUB_RULE}_code'
    reason_code_col = f'{STUB_REASON}_code'

    expected_primary = {}
    expected_also = {}
    rule_by_code = {}
    for _l1code, node in json.loads(
            (TAXONOMY_DIR / f'{MOD_RULE}.json').read_text(encoding='utf-8')).items():
        for _lc, _leaf in node.get('children', {}).items():
            _n = re.match(r'^(\d+)\s', _leaf['label'])
            if not _n:
                continue                      # the authored NA leaf has no number
            _rule = next(r for r in RULES if r['n'] == int(_n.group(1)))
            rule_by_code[_lc] = _rule
            expected_primary[_lc] = _rule['primary']
            expected_also[_lc] = set(_rule['also'])

    out = tagged.merge(
        truth[[ID_COL, RC_COL, 'rc_key', CR1_COL, CR2_COL, 'payload_chars']],
        on=ID_COL, how='left', validate='one_to_one')
    assert out['rc_key'].notna().all(), 'a tagged ticket has no ground truth row'
    # Rename before anything else touches them - see the note in section 0.
    out = out.rename(columns={RC_COL: RC_LABEL, CR1_COL: CR1_LABEL, CR2_COL: CR2_LABEL})

    out['rc_actual']        = out['rc_key']
    out['rule_is_na']       = out[rule_code_col].astype(str).str.strip() == NA_RULE_CODE
    out['rule_n']           = out[rule_code_col].map(
        lambda c: rule_by_code.get(str(c).strip(), {}).get('n'))
    out['expected_primary'] = out[rule_code_col].map(
        lambda c: expected_primary.get(str(c).strip(), ''))
    out['expected_also']    = out[rule_code_col].map(
        lambda c: sorted(expected_also.get(str(c).strip(), set())))
    out['is_mappable']      = out['rc_actual'].isin(PRIMARY_CODES)
    out['conf_strict']      = (out['expected_primary'] != '') & \
                              (out['rc_actual'] == out['expected_primary'])
    out['conf_loose']       = out['conf_strict'] | out.apply(
        lambda r: r['rc_actual'] in set(r['expected_also']), axis=1)
    out['reason_is_none']   = out[reason_code_col].astype(str).str.strip().isin(NO_REASON_CODES)
    return out


def conformance_summary(df: pd.DataFrame) -> dict:
    """Conformance rates, each with the bound that actually applies to it.

    The three rates have DIFFERENT denominators and therefore different maxima -
    pairing the wrong bound with the wrong rate is how you conclude you are done
    when you are not:

      conformance_mappable  restricted to tickets whose code a rule claims as
                            primary.  Maximum 100%.  This is the headline.
      conformance_strict    over every rule-tagged ticket.  A ticket whose code no
                            rule claims can never conform, so the maximum is the
                            mappable share of rule-tagged tickets.
      MATRIX_CODE_COVERAGE  share of ALL tickets whose code a rule claims. That is
                            a property of the matrix, not a bound on either rate.
    """
    tagged = df[~df['rule_is_na']]
    mappable = tagged[tagged['is_mappable']]
    mappable_share = float(tagged['is_mappable'].mean()) if len(tagged) else float('nan')
    return {
        'n_total': len(df),
        'n_rule_tagged': len(tagged),
        'n_rule_na': int(df['rule_is_na'].sum()),
        'rule_na_rate': float(df['rule_is_na'].mean()),
        'n_mappable': len(mappable),
        'mappable_share_of_tagged': mappable_share,
        'conformance_strict': float(tagged['conf_strict'].mean()) if len(tagged) else float('nan'),
        'conformance_loose': float(tagged['conf_loose'].mean()) if len(tagged) else float('nan'),
        'conformance_mappable': float(mappable['conf_strict'].mean()) if len(mappable)
        else float('nan'),
        'matrix_code_coverage': MATRIX_CODE_COVERAGE,
    }


def print_conformance(summary: dict, label: str) -> None:
    print(f'--- {label} ---')
    print(f'  tickets                        {summary["n_total"]:,}')
    print(f'  tagged with a rule             {summary["n_rule_tagged"]:,} '
          f'({1 - summary["rule_na_rate"]:.1%})')
    print(f'  no applicable rule ({NA_RULE_CODE})   {summary["n_rule_na"]:,} '
          f'({summary["rule_na_rate"]:.1%})')
    print(f'  of the rule-tagged, mappable   {summary["n_mappable"]:,} '
          f'({summary["mappable_share_of_tagged"]:.1%}) - the rest cannot conform at all')
    print()
    print(f'  CONFORMANCE AMONG MAPPABLE     {summary["conformance_mappable"]:.1%} '
          f'of a possible 100.0%     <- HEADLINE')
    print(f'  conformance strict, all tagged {summary["conformance_strict"]:.1%} '
          f'of a possible {summary["mappable_share_of_tagged"]:.1%}')
    print(f'  conformance loose (+also)      {summary["conformance_loose"]:.1%}')
    print()
    print(f'  context: {summary["matrix_code_coverage"]:.1%} of all 2026 tickets carry a code '
          f'some rule claims as primary.')
    print(f'           That is a property of the matrix. It does NOT bound the headline, '
          f'which is already')
    print(f'           restricted to mappable tickets. Headroom on the headline: '
          f'{1 - summary["conformance_mappable"]:.1%}.')


pr = derive_conformance(read_stage_output(probe_out), to_classify)

# 1. blank rate
_blank = (pr[f'{STUB_RULE}_code'].isna() |
          (pr[f'{STUB_RULE}_code'].astype(str).str.strip() == '')).mean()
print(f'blank rule codes: {_blank:.2%}')
assert _blank <= 0.005, f'{_blank:.2%} blank rule codes - the pipeline is losing records'

# 2. the applies tripwire
_says_yes = pr[f'{STUB_RULE}_applies'].astype(str).str.strip().str.lower() == 'yes'
_agree = (_says_yes != pr['rule_is_na']).mean()
print(f'{STUB_RULE}_applies agrees with code == {NA_RULE_CODE}: {_agree:.1%}')
if _agree < 1.0:
    print(pr.loc[_says_yes == pr['rule_is_na'],
                 [ID_COL, f'{STUB_RULE}_code', f'{STUB_RULE}_applies']].head(10)
          .to_string(index=False))
assert _agree >= 0.98, (
    f'the model disagrees with itself on {1 - _agree:.1%} of tickets - it is choosing a code '
    f'and answering the applies question as two unrelated decisions')

# 3. first real conformance read
PROBE_SUMMARY = conformance_summary(pr)
print()
print_conformance(PROBE_SUMMARY, f'probe, {len(pr)} tickets')
assert PROBE_SUMMARY['conformance_mappable'] >= 0.40, (
    f'conformance among mappable tickets is {PROBE_SUMMARY["conformance_mappable"]:.1%}. '
    f'Below 40% the prompt is wrong, not the data. Read the confusion pairs below before '
    f'spending the main run budget.')

In [ ]:
# Where the model and the agent disagree most, with the model's own reasoning.
_bad = pr[(~pr['rule_is_na']) & pr['is_mappable'] & ~pr['conf_strict']]
_pairs = (_bad.groupby([f'{STUB_RULE}_label', RC_LABEL]).size()
          .sort_values(ascending=False).head(15))
print('Top 15 confusion pairs (model said rule X, agent logged code Y):\n')
for (_rule_label, _code), _n in _pairs.items():
    _exp = _bad.loc[_bad[f'{STUB_RULE}_label'] == _rule_label, 'expected_primary'].iloc[0]
    print(f'  {_n:>4}  {_rule_label[:58]:<58}')
    print(f'        expected {_exp!r} but agent logged {_code!r}')

print('\n\nThree examples with the model reasoning:\n')
for _, r in _bad.head(3).iterrows():
    print('=' * 78)
    print(f'{r[ID_COL]}  model: {r[f"{STUB_RULE}_label"]}')
    print(f'  expected code : {r["expected_primary"]}')
    print(f'  agent logged  : {r["rc_actual"]}')
    print(f'  why           : {r[f"{STUB_RULE}_justification"]}')
    print(f'  evidence      : {str(r[f"{STUB_RULE}_evidence"])[:250]}')

### Decision point

`STOP_AFTER_PROBE` is `True` by default. Before clearing it, check:

- **Conformance among mappable** is in a range you believe. Very high (>90%) on this data would be
  suspicious, not reassuring — it would suggest the model found a shortcut.
- **The NA rate clears its floor.** The floor is the share of tickets whose resolution code no
  rule claims at all (`advice request` + `book`, about 12–13%). Section 18.1 computes it. Below
  the floor the model is force-fitting rules onto tickets nothing can cover — which also hides
  the matrix gap that section 16 exists to find.
- **Section 13(b) shows a low false-positive rate on the re-evaluation disclaimer.** The first
  probe run read a standing boilerplate sentence as a finding on 44% of the tickets containing
  it. If that number is still above ~20%, fix the cue before spending the main run budget.
- **The confusion pairs above are disagreements you can explain.** If the top pair is
  "too recent" vs "already delivered", that is a genuine boundary. If it is "journal not covered"
  vs "data correction", something is wrong.
- **The evidence quotes are real.** Section 11 asserted 80%; skim a few here too.

Set it to `False`, add the date and what you checked, and continue.

In [ ]:
STOP_AFTER_PROBE = False   # set False once the numbers above have been read

if STOP_AFTER_PROBE:
    raise SystemExit(
        'Stopping after the probe. Review the conformance figures, the NA rate and the '
        'confusion pairs above, then set STOP_AFTER_PROBE = False and re-run from here. '
        f'The main run costs about ${SAMPLE_N * 0.0065 * len(TAGGING_MODULES):,.0f}.')
print('Proceeding to the main run.')

## 13. Template phrase audit *(free)*

Agents paste template wording verbatim, and a few phrases are near-deterministic tells for a
single resolution code. This cell is **not a classifier** — it is a diagnostic on the one we
built. For each phrase it reports how concentrated the resolution code is when the phrase is
present, and whether the model tagged the matching rule.

If precision against the resolution code is 95% but the model only tagged the matching rule 60% of
the time, the prompt is missing a cue that is sitting in plain text.

**Why there is no full template matcher.** The templates workbook has no consistent key: in
`Missing Content Template` the scenario label sits in column A on a *different row* from the body
in column B, and the offset varies. Building a reliable label→body map is manual data entry
wearing a costume. More importantly, the phrases that would matter most do not discriminate —
"re-evaluation" appears in 9,158 tickets of which the plurality are `advice request`, not
`reevaluation`, and "discontinued" behaves the same way. A matcher would be confident exactly
where the model is already right and silent where it is not.

In [ ]:
import math


def _wilson_lo(k: int, n: int, z: float = 1.96) -> float:
    """Lower bound of the Wilson score interval - so a threshold check on a small
    sample does not cry wolf. At n=88 a measured 24% has a 95% CI of roughly
    +/- 9pp, which is wider than most of the deltas worth acting on."""
    if not n:
        return float('nan')
    p = k / n
    d = 1 + z * z / n
    centre = p + z * z / (2 * n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return max(0.0, (centre - half) / d)


def run_prompt_audits(_df: pd.DataFrame, label: str = 'probe') -> dict:
    """Three prompt diagnostics. Safe to run on the probe or on the full run.

    (a) phrases that SHOULD produce a rule - are strong textual cues reaching it?
    (b) boilerplate that must NOT produce a rule - is a disclaimer read as a finding?
    (c) the rule 5 / rule 7 boundary - the largest single error in the first probe.

    Each threshold check uses the lower bound of a 95% Wilson interval, so it fires
    on evidence rather than on a point estimate from a small stratum.
    """
    out = {}
    print('=' * 78)
    print(f'PROMPT AUDITS - {label} (n={len(_df):,})')
    print('=' * 78)

    # ---- (a) phrases that should lead to a rule ------------------------
    rule_for_code = {}
    for r in RULES:
        rule_for_code.setdefault(r['primary'], []).append(r['n'])

    rows = []
    for phrase, expect in TEMPLATE_PROBES:
        hit_all = to_classify[to_classify[TEXT_COL].str.contains(phrase, case=False, na=False)]
        if hit_all.empty:
            continue
        hit = _df[_df[ID_COL].isin(set(hit_all[ID_COL]))]
        want = set(rule_for_code.get(expect, []))
        rows.append({'phrase': phrase, 'n_corpus': len(hit_all),
                     'code_precision': (hit_all['rc_key'] == expect).mean(),
                     'n_tagged': len(hit),
                     'rule_recall': hit['rule_n'].isin(want).mean() if len(hit) else float('nan'),
                     'expected_code': expect})
    out['phrase'] = pd.DataFrame(rows)
    print('\n(a) Phrases that should lead to a rule\n')
    print(out['phrase'].to_string(index=False,
                                  formatters={'code_precision': '{:.0%}'.format,
                                              'rule_recall': '{:.0%}'.format}))
    weak = out['phrase'][(out['phrase']['n_tagged'] >= 20)
                         & (out['phrase']['code_precision'] >= 0.85)
                         & (out['phrase']['rule_recall'] < 0.85)]
    weak = weak[[_wilson_lo(int(r['rule_recall'] * r['n_tagged']), int(r['n_tagged'])) < 0.70
                 for _, r in weak.iterrows()]]
    if len(weak):
        print(f'\n{len(weak)} phrase(s) where a strong code signal is not reaching the rule, '
              f'beyond sampling noise:')
        print(weak[['phrase', 'expected_code', 'code_precision', 'rule_recall']]
              .to_string(index=False))
    else:
        print('\nNo phrase where a strong code signal demonstrably fails to reach its rule.')

    # ---- (b) boilerplate that must NOT lead to a rule ------------------
    txt = to_classify.set_index(ID_COL)[TEXT_COL].reindex(_df[ID_COL]).fillna('')
    boiler = txt.str.contains(BOILERPLATE_RE, case=False, regex=True).to_numpy()
    real = txt.str.contains(REAL_REEVAL_RE, case=False, regex=True).to_numpy()
    is_re = (_df[f'{STUB_RULE}_code'].astype(str).str.strip() == REEVAL_CODE).to_numpy()
    bo = boiler & ~real

    out['false_positive'] = pd.DataFrame([
        {'group': 'disclaimer only, no real claim', 'n': int(bo.sum()),
         'tagged': float(is_re[bo].mean()) if bo.sum() else float('nan'), 'want': 'low'},
        {'group': 'source-specific re-evaluation claim', 'n': int(real.sum()),
         'tagged': float(is_re[real].mean()) if real.sum() else float('nan'), 'want': 'high'},
        {'group': 'neither', 'n': int((~boiler & ~real).sum()),
         'tagged': float(is_re[~boiler & ~real].mean()), 'want': 'low'},
    ])
    print('\n\n(b) Boilerplate that must NOT lead to a rule\n')
    print(out['false_positive'].to_string(index=False, formatters={'tagged': '{:.0%}'.format}))
    fp, tp = out['false_positive'].iloc[0]['tagged'], out['false_positive'].iloc[1]['tagged']
    fp_lo = _wilson_lo(int(is_re[bo].sum()), int(bo.sum()))
    print(f'\nfalse positive {fp:.0%} (95% CI lower bound {fp_lo:.0%}); was 44% before any cue')
    print(f'true positive  {tp:.0%}; was 83%. The cue must not cost recall here.')
    if fp_lo > 0.25:
        print('\n  Beyond noise: the disclaimer is still being read as a finding. Strengthen')
        print('  RULE_NEGATIVE_CUES[2] in section 4 and re-run from section 9.')
    elif tp < 0.70:
        print('\n  Over-corrected: genuine re-evaluation statements are now being missed.')
    else:
        print('\n  Within noise of the target. Remaining false positives are worth reading in')
        print('  18.7 rather than tuning further against one sample.')

    # ---- (c) the rule 5 / rule 7 boundary ------------------------------
    rows = []
    for code, want in ((PREMATURE_CODE, 'invalid-premature'),
                       (PIPELINE_CODE, 'invalid-recently delivered in obii')):
        t = _df[_df[f'{STUB_RULE}_code'] == code]
        a = _df[_df['rc_actual'] == want]
        rows.append({'rule': code, 'expected_code': want, 'n_tagged': len(t),
                     'precision': t['rc_actual'].eq(want).mean() if len(t) else float('nan'),
                     'n_actual': len(a),
                     'recall': a[f'{STUB_RULE}_code'].eq(code).mean() if len(a) else float('nan')})
    out['boundary'] = pd.DataFrame(rows)
    print('\n\n(c) The rule 5 / rule 7 boundary\n')
    print(out['boundary'].to_string(index=False, formatters={'precision': '{:.0%}'.format,
                                                             'recall': '{:.0%}'.format}))
    print('\nprobe 1 (no cues) : CSDVR1.5 70% / 49%    CSDVR1.7 37% / 95%')
    print('probe 3 (cued)    : CSDVR1.5 77% / 89%    CSDVR1.7 73% / 97%')
    if out['boundary'].iloc[1]['precision'] < 0.50:
        print('\n  CSDVR1.7 is still absorbing premature tickets. Strengthen '
              'RULE_NEGATIVE_CUES[7].')
    else:
        print('\n  The boundary is holding. Some genuine OBII tickets are simply not')
        print('  distinguishable from the transcript - Opsbank and OBII never appear in')
        print('  customer-facing replies - so CSDVR1.7 recall is capped by the data, not')
        print('  by the prompt.')
    return out


# Phrases measured against this corpus; each is concentrated on one resolution code.
TEMPLATE_PROBES = [
    ('allow 6 weeks after the publication date', 'invalid-premature'),
    ('indexation process generally takes several weeks', 'invalid-premature'),
    ('evaluation process takes between 2-3 months', 'invalid-book'),
    ('Article in Press', 'invalid-aip'),
    ('Title Suggestion', 'invalid-title not covered'),
    ('authenticate', 'invalid-not authentic'),
]
BOILERPLATE_RE = r'may be subject to re-?evaluation at any (?:point|time)'
REAL_REEVAL_RE = (r'(?:currently|is|being)\s+(?:undergoing|under)\s+re-?evaluation'
                  r'|re-?evaluation by the (?:scopus )?content selection')
REEVAL_CODE = f'{PREFIX_RULE}1.2'
PREMATURE_CODE, PIPELINE_CODE = f'{PREFIX_RULE}1.5', f'{PREFIX_RULE}1.7'

_audits = run_prompt_audits(pr, 'probe')
PHRASE_AUDIT = _audits['phrase']
FALSE_POSITIVE_AUDIT = _audits['false_positive']
BOUNDARY_AUDIT = _audits['boundary']


## 14. Main run — 12,000 stratified tickets *(~$156, ~2.2 h)*

Proportional by resolution code with a floor of 300 per code, so all 15 stay estimable — including
`Invalid- Out of coverage`, which has only 61 tickets in the whole corpus and therefore contributes
all of them.

12,000 gives roughly ±0.9pp at 95% confidence on the headline conformance rate, which is far
tighter than the measurement is meaningful given that the ground truth is agent-assigned. Running
all 50,721 costs 4× more and buys precision nobody will use. Section 17 is there if a per-ticket
tag on every ticket is an operational requirement.

The run is resumable. If it dies, re-run this cell: `run_stage` will find the checkpoint, the
fingerprints will match, and only the outstanding batches are dispatched.

In [ ]:
main_sample = stratified_sample(to_classify, RC_COL, SAMPLE_N,
                                min_per=SAMPLE_MIN_PER_CODE, seed=RANDOM_SEED)
main_sample[LLM_COLS].to_csv(SAMPLE_CSV, index=False)

print(f'Main sample: {len(main_sample):,} tickets '
      f'(target {SAMPLE_N:,}, floor {SAMPLE_MIN_PER_CODE}/code)')
_cmp = pd.DataFrame({
    'sample': main_sample[RC_COL].value_counts(),
    'corpus': to_classify[RC_COL].value_counts()})
_cmp['sample_share'] = (_cmp['sample'] / _cmp['sample'].sum()).map('{:.1%}'.format)
_cmp['corpus_share'] = (_cmp['corpus'] / _cmp['corpus'].sum()).map('{:.1%}'.format)
print(_cmp.to_string())
assert main_sample[RC_COL].nunique() == to_classify[RC_COL].nunique()
assert main_sample[ID_COL].is_unique

print(f'\nEstimated cost: ${len(main_sample) * 0.0065 * len(TAGGING_MODULES):,.0f}')

In [ ]:
main_out = run_stage(PROFILE_TAG, modules=TAGGING_MODULES,
                     input_path=SAMPLE_CSV, output_path=MAIN_OUT,
                     label=f'MAIN RUN - {len(main_sample):,} tickets, rule + reason')

## 15. Merge and derive

The metadata is joined to the model output **here, for the first time**. Everything before this
point was blind.

One naming trap is worth restating: `resolution_` is in `rca_pipeline.utils._STRUCTURAL_PREFIXES`,
so any column called `resolution_something` would be silently dropped if this frame were ever fed
back through the pipeline. All derived columns are `rc_*` or `conf_*`.

In [ ]:
TAGGED = derive_conformance(read_stage_output(main_out), to_classify)
TAGGED['rule_label'] = TAGGED[f'{STUB_RULE}_label']
TAGGED['reason_label'] = TAGGED[f'{STUB_REASON}_label']

assert len(TAGGED) == len(main_sample), f'{len(TAGGED)} rows back, expected {len(main_sample)}'
assert not [c for c in TAGGED.columns if c.startswith('resolution_')], (
    'a derived column starts with resolution_ and would be stripped on a re-run')

MAIN_SUMMARY = conformance_summary(TAGGED)
print_conformance(MAIN_SUMMARY, f'main run, {len(TAGGED):,} tickets')

print(f'\nrejection reason = "no reason stated": {int(TAGGED["reason_is_none"].sum()):,} '
      f'({TAGGED["reason_is_none"].mean():.1%})')
print(f'distinct rules used  : {TAGGED[f"{STUB_RULE}_code"].nunique()} of {len(RULE_LEAVES)}')
print(f'distinct reasons used: {TAGGED[f"{STUB_REASON}_code"].nunique()} of {len(REASON_LEAVES)}')

## 16. Gap pass — draft the rules the matrix is missing *(~$15)*

Every ticket tagged `CSDVR4.1` is a ticket the 24 rules do not describe. Section 18.5 quantifies
where those cluster; this section asks the model to draft the rule that *would* cover them, in the
matrix's own phrasing.

The sample is stratified by **discovered rejection reason**, not by resolution code, because the
question is "what kind of refusal has no rule", and the rejection reason is the finer-grained
answer to that.

Proposals are aggregated per rejection reason rather than reported per ticket: one draft rule per
gap, with the volume it would cover and three example tickets. **This is LLM-drafted text and
needs CSD sign-off** — it is a starting point for a conversation with the content team, not a
policy change.

In [ ]:
na_pool = TAGGED[TAGGED['rule_is_na']].copy()
print(f'Tickets with no applicable rule: {len(na_pool):,} of {len(TAGGED):,} '
      f'({len(na_pool) / len(TAGGED):.1%})')
print('\nBy discovered rejection reason:')
print(na_pool['reason_label'].value_counts().head(20).to_string())

# Always defined, so the next two cells no-op cleanly instead of raising NameError.
prop_sample = na_pool.iloc[:0]
if len(na_pool) == 0:
    print('\nNothing to propose - every ticket matched a rule. Skip to section 18.')
else:
    prop_sample = stratified_sample(na_pool, f'{STUB_REASON}_code',
                                    min(PROPOSAL_N, len(na_pool)),
                                    min_per=10, seed=RANDOM_SEED)
    # The pipeline output carries the conversation through, but take the canonical
    # text from to_classify so the gap pass reads exactly what the tagging pass read.
    prop_sample = prop_sample.merge(to_classify[[ID_COL, TEXT_COL]], on=ID_COL,
                                    how='left', suffixes=('_drop', ''))
    prop_sample[LLM_COLS].to_csv(PROPOSAL_CSV, index=False)
    print(f'\nProposal sample: {len(prop_sample):,} tickets across '
          f'{prop_sample[f"{STUB_REASON}_code"].nunique()} rejection reasons')
    print(f'Estimated cost: ${len(prop_sample) * 0.0065:,.0f}')


In [ ]:
prop_out = None
if len(prop_sample):
    prop_out = run_stage(PROFILE_PROP, modules=[MOD_PROPOSAL],
                         input_path=PROPOSAL_CSV, output_path=PROPOSAL_OUT,
                         label=f'GAP PASS - {len(prop_sample):,} uncovered tickets')
else:
    print('No uncovered tickets to draft rules for - skipping the gap pass.')


In [ ]:
PROPOSED_RULES = pd.DataFrame()
if prop_out is None:
    raise SystemExit('No gap pass ran; PROPOSED_RULES stays empty. '
                     'Continue at section 17.')

props = read_stage_output(prop_out).merge(
    prop_sample[[ID_COL, f'{STUB_REASON}_code', 'reason_label', RC_LABEL]],
    on=ID_COL, how='left')

def _modal(series):
    vals = [v for v in series.map(norm_label) if v]
    return Counter(vals).most_common(1)[0][0] if vals else ''


rows = []
for (_code, _label), g in props.groupby([f'{STUB_REASON}_code', 'reason_label']):
    _texts = [norm_prose(t) for t in g[f'{STUB_PROPOSAL}_rule_text'] if norm_prose(t)]
    if not _texts:
        continue
    # The modal proposal is the one the model converged on independently across tickets;
    # a reason where every ticket produced a different wording is not a coherent gap.
    _counts = Counter(_texts)
    _top, _top_n = _counts.most_common(1)[0]
    rows.append({
        'Request Type': _modal(g[f'{STUB_PROPOSAL}_request_type']),
        'Proposed Validation Rule': _top,
        'Primary Resolution Code (observed)': _modal(g[RC_LABEL]),
        'Primary Resolution Code (model)': _modal(g[f'{STUB_PROPOSAL}_expected_code']),
        'Rejection reason': _label,
        'reason_code': _code,
        'n_sampled': len(g),
        'agreement': _top_n / len(g),
        'n_uncovered_in_run': int((na_pool[f'{STUB_REASON}_code'] == _code).sum()),
        'examples': '; '.join(g[ID_COL].head(3)),
        'evidence': ' | '.join(str(e)[:160] for e in g[f'{STUB_PROPOSAL}_evidence'].head(2)),
    })

PROPOSED_RULES = (pd.DataFrame(rows)
                  .sort_values('n_uncovered_in_run', ascending=False)
                  .reset_index(drop=True))
print(f'{len(PROPOSED_RULES)} candidate rules, covering '
      f'{int(PROPOSED_RULES["n_uncovered_in_run"].sum()):,} of the {len(na_pool):,} '
      f'uncovered tickets in this run.\n')
for _, r in PROPOSED_RULES.head(12).iterrows():
    print(f'[{r["n_uncovered_in_run"]:>5} tickets, {r["agreement"]:.0%} agreement across '
          f'{r["n_sampled"]} samples]')
    print(f'  type    : {r["Request Type"]}')
    print(f'  rule    : {r["Proposed Validation Rule"]}')
    print(f'  code    : observed {r["Primary Resolution Code (observed)"]!r} / '
          f'model says {r["Primary Resolution Code (model)"]!r}')
    print(f'  reason  : {r["Rejection reason"]}')
    print(f'  examples: {r["examples"]}\n')
print('LLM-drafted. Needs CSD sign-off against the current CCG/CAR before any operational use.')

## 17. Optional — the full corpus *(~$660, ~9 h)*

Only needed if the deliverable is a tag on **every** 2026 ticket rather than an analysis. The
12,000-ticket sample already answers every question in Section 18 to within a percentage point.

Flip the flag, run, and Section 18 will pick up `TAGGED` from the full run instead.

In [ ]:
RUN_FULL_CORPUS = False

if RUN_FULL_CORPUS:
    full_out = run_stage(PROFILE_TAG, modules=TAGGING_MODULES,
                         input_path=STAGE_INPUT,
                         output_path=OUT_DIR / 'csd_2026_rules_and_reasons_full.xlsx',
                         label=f'FULL CORPUS - {len(to_classify):,} tickets')
    TAGGED = derive_conformance(read_stage_output(full_out), to_classify)
    TAGGED['rule_label'] = TAGGED[f'{STUB_RULE}_label']
    TAGGED['reason_label'] = TAGGED[f'{STUB_REASON}_label']
    MAIN_SUMMARY = conformance_summary(TAGGED)
    print_conformance(MAIN_SUMMARY, f'full corpus, {len(TAGGED):,} tickets')
else:
    print(f'Skipped. Analysis runs on the {len(TAGGED):,}-ticket sample.')
    print(f'Set RUN_FULL_CORPUS = True to tag all {len(to_classify):,} classifiable tickets '
          f'(~${len(to_classify) * 0.0065 * len(TAGGING_MODULES):,.0f}).')

## 18. Analysis

Seven cells. The first establishes what fraction of tickets got a usable tag; the middle ones
quantify the three pairwise relationships; the last two are the payoff — the three-way table and a
sample of disagreements for human judgement.

**Conformance is not accuracy.** It measures agreement between the rule the model read out of the
ticket and the code the agent chose. `Invalid- Others` alone accounts for ~3,900 tickets and is a
wastebasket. Where the two disagree, either can be wrong, and Section 18.7 is the only way to
tell which.

### 18.1 Coverage and NA rates

In [ ]:
cov = pd.DataFrame([{
    'metric': 'tickets in the 2026 file', 'n': len(corpus), 'pct': 1.0}, {
    'metric': 'excluded as unclassifiable', 'n': len(excluded),
    'pct': len(excluded) / len(corpus)}, {
    'metric': 'classifiable', 'n': len(to_classify),
    'pct': len(to_classify) / len(corpus)}, {
    'metric': 'tagged in this run', 'n': len(TAGGED), 'pct': len(TAGGED) / len(corpus)}, {
    'metric': '  of which a rule applies', 'n': int((~TAGGED['rule_is_na']).sum()),
    'pct': float((~TAGGED['rule_is_na']).mean())}, {
    'metric': f'  of which no rule applies ({NA_RULE_CODE})',
    'n': int(TAGGED['rule_is_na'].sum()), 'pct': float(TAGGED['rule_is_na'].mean())}, {
    'metric': '  of which no reason was stated', 'n': int(TAGGED['reason_is_none'].sum()),
    'pct': float(TAGGED['reason_is_none'].mean())}])
cov['pct'] = cov['pct'].map('{:.1%}'.format)
COVERAGE_TABLE = cov
print(cov.to_string(index=False))

# Expected NA floor, computed rather than guessed: tickets whose resolution code no
# rule claims either as a primary OR as an also-maps-to have no rule that can apply.
# ('discontinued title' is deliberately NOT in the floor - rule 8 lists it as an
# also-maps-to, so a rule legitimately applies to it.)
_no_rule_codes = {c for c in TAGGED['rc_actual'].unique()
                  if c not in PRIMARY_CODES and c not in ALSO_CODES}
NA_FLOOR = float(TAGGED['rc_actual'].isin(_no_rule_codes).mean())
_na = TAGGED['rule_is_na'].mean()
print(f'\nNA rate {_na:.1%} against a floor of {NA_FLOOR:.1%}')
print(f'  (codes no rule claims at all: {", ".join(sorted(_no_rule_codes))})')
if _na < NA_FLOOR * 0.85:
    print('  BELOW THE FLOOR: the model is force-fitting rules onto tickets that no rule can')
    print('  cover. This does not just cost accuracy - it hides the largest gap in the matrix,')
    print('  because those tickets never reach the NA code and so never reach section 16.')
    print('  Check the per-code NA rates below and section 18.5.')
elif _na > 0.40:
    print('  Over 40%: the model is avoiding commitment. Check the confusion output in 18.7.')
else:
    print('  At or above the floor, as expected: some tickets whose code IS claimed by a rule')
    print('  still legitimately match none of them.')

# NA rate per resolution code. The unmapped codes should sit near the top; if one of
# them is near zero, the model is absorbing it into some rule it does not belong to.
print('\nNA rate by resolution code (the unmapped ones should be highest):')
_na_by_code = (TAGGED.groupby('rc_actual')
               .agg(n=('rule_is_na', 'size'), na_rate=('rule_is_na', 'mean')))
_na_by_code['no_rule_can_apply'] = [c in _no_rule_codes for c in _na_by_code.index]
_na_by_code = _na_by_code.sort_values('na_rate', ascending=False)
print(_na_by_code.to_string(formatters={'na_rate': '{:.0%}'.format}))
_absorbed = _na_by_code[_na_by_code['no_rule_can_apply'] & (_na_by_code['na_rate'] < 0.60)]
if len(_absorbed):
    print(f'\n  WARNING: {len(_absorbed)} code(s) that no rule can cover are being given a rule')
    print(f'  most of the time: {", ".join(_absorbed.index)}. Section 18.5 will under-report')
    print('  the matrix gap by roughly that much.')

print('\nNA rate by contact reason (where the matrix has holes):')
_by_cr = (TAGGED.groupby(CR1_LABEL)
          .agg(n=('rule_is_na', 'size'), na_rate=('rule_is_na', 'mean'))
          .query('n >= 20').sort_values('na_rate', ascending=False))
_by_cr['na_rate'] = _by_cr['na_rate'].map('{:.1%}'.format)
print(_by_cr.to_string())


### 18.2 Validation rule × resolution code

In [ ]:
RULE_X_CODE = pd.crosstab(TAGGED['rule_label'], TAGGED[RC_LABEL])
RULE_X_CODE_ROWPCT = pd.crosstab(TAGGED['rule_label'], TAGGED[RC_LABEL], normalize='index')

pd.set_option('display.width', 250, 'display.max_columns', 40)
print('Counts:\n')
print(RULE_X_CODE.to_string())

print('\n\nRow-normalised - given the model said rule X, which code did the agent log?\n')
_rp = RULE_X_CODE_ROWPCT.copy()
_rp.columns = [c.replace('Invalid-', '').strip()[:14] for c in _rp.columns]
print(_rp.map(lambda v: f'{v:.0%}' if v >= 0.005 else '.').to_string())

print('\n\nPer rule: does the expected primary code actually dominate?\n')
_rows = []
for _code, _lab in sorted(RULE_LEAVES.items()):
    _sub = TAGGED[TAGGED[f'{STUB_RULE}_code'] == _code]
    if _sub.empty:
        continue
    _exp = _sub['expected_primary'].iloc[0]
    _top = _sub['rc_actual'].value_counts()
    # Where a rule's errors concentrate matters as much as its error rate. A rule
    # whose misses pile onto a single resolution code is either fighting a
    # neighbouring rule over a boundary, or standing in for a rule the matrix lacks.
    _err = _sub[~_sub['conf_strict']]['rc_actual'].value_counts()
    _rows.append({'code': _code, 'rule': _lab[:52], 'n': len(_sub),
                  'expected': _exp or '(NA leaf)',
                  'modal_actual': _top.index[0], 'modal_share': _top.iloc[0] / len(_sub),
                  'strict': _sub['conf_strict'].mean(), 'loose': _sub['conf_loose'].mean(),
                  'modal_error': _err.index[0] if len(_err) else '',
                  'modal_error_share': (_err.iloc[0] / _err.sum()) if len(_err) else 0.0,
                  'estimable': len(_sub) >= 30})
PER_RULE = pd.DataFrame(_rows).sort_values('n', ascending=False)
print(PER_RULE.to_string(index=False, formatters={
    'modal_share': '{:.0%}'.format, 'strict': '{:.0%}'.format, 'loose': '{:.0%}'.format}))
print(f'\n{int((~PER_RULE["estimable"]).sum())} rule(s) have fewer than 30 tickets and their '
      f'rates are not estimable.')

# Rules whose errors are concentrated rather than scattered. Read these before
# concluding the model is simply wrong: a rule absorbing one specific code often
# means the matrix has no rule for that code.
# The NA leaf has no expected primary code, so its 'strict' rate is 0 by
# construction - excluded, or it heads this table every time and means nothing.
_conc = PER_RULE[(PER_RULE['code'] != NA_RULE_CODE) & (PER_RULE['estimable'])
                 & (PER_RULE['strict'] < 0.85) & (PER_RULE['modal_error_share'] >= 0.35)]
print('\n\nRules whose errors concentrate on ONE resolution code '
      '(boundary problem, or a missing rule):\n')
if len(_conc):
    print(_conc[['code', 'rule', 'n', 'strict', 'modal_error', 'modal_error_share']]
          .to_string(index=False, formatters={'strict': '{:.0%}'.format,
                                              'modal_error_share': '{:.0%}'.format}))
    print('\nIf the modal error code has its own rule, this is a boundary to sharpen with a')
    print('negative cue in RULE_NEGATIVE_CUES. If it does not - or its rules are far narrower')
    print('than the code they carry, as with the two AiP rules - the matrix is missing a rule,')
    print('and section 16 will NOT surface it, because those tickets get a plausible rule')
    print('instead of the NA code and so never enter the gap sample. Add them by hand.')
else:
    print('  None: every rule scatters its errors.')


### 18.3 Conformance

In [ ]:
print_conformance(MAIN_SUMMARY, f'{len(TAGGED):,} tickets')

# The three rates and the bound that actually applies to each. Getting this pairing
# wrong is the single easiest way to misread this notebook: 'matrix code coverage'
# and 'conformance among mappable' can land on similar-looking numbers by pure
# coincidence, and reading one as the ceiling of the other says you are finished
# when you still have all your headroom.
print('\n' + '=' * 78)
print('WHICH BOUND APPLIES TO WHICH RATE')
print('=' * 78)
_ms = MAIN_SUMMARY['mappable_share_of_tagged']
print(f'  conformance among mappable   {MAIN_SUMMARY["conformance_mappable"]:>6.1%}   '
      f'max 100.0%   headroom {1 - MAIN_SUMMARY["conformance_mappable"]:.1%}')
print(f'  conformance strict           {MAIN_SUMMARY["conformance_strict"]:>6.1%}   '
      f'max {_ms:>5.1%}   (a rule-tagged ticket whose code no rule claims')
print(f'  {"":<30}                cannot conform, whatever the model does)')
print(f'  matrix code coverage         {MATRIX_CODE_COVERAGE:>6.1%}   '
      f'a property of the MATRIX over all tickets -')
print(f'  {"":<30}                not a bound on either rate above')
print(f'\n  identity: {_ms:.3f} x {MAIN_SUMMARY["conformance_mappable"]:.3f} = '
      f'{_ms * MAIN_SUMMARY["conformance_mappable"]:.3f} = conformance strict')

_rows = []
for _code in sorted(TAGGED['rc_actual'].unique()):
    _sub = TAGGED[(TAGGED['rc_actual'] == _code) & (~TAGGED['rule_is_na'])]
    _all = TAGGED[TAGGED['rc_actual'] == _code]
    _rows.append({'resolution_code': _code, 'n': len(_all),
                  'na_rate': _all['rule_is_na'].mean(),
                  'mappable': _code in PRIMARY_CODES,
                  'strict': _sub['conf_strict'].mean() if len(_sub) else float('nan'),
                  'loose': _sub['conf_loose'].mean() if len(_sub) else float('nan')})
PER_CODE = pd.DataFrame(_rows).sort_values('n', ascending=False)
print('\n\nPer resolution code:\n')
print(PER_CODE.to_string(index=False, formatters={
    'na_rate': '{:.0%}'.format, 'strict': '{:.0%}'.format, 'loose': '{:.0%}'.format}))

CONFORMANCE_TABLE = pd.DataFrame([
    {'metric': 'CONFORMANCE AMONG MAPPABLE (headline)',
     'value': f'{MAIN_SUMMARY["conformance_mappable"]:.1%}',
     'maximum possible': '100.0%',
     'note': 'Restricted to tickets whose resolution code some rule claims as primary.'},
    {'metric': 'conformance strict, all rule-tagged',
     'value': f'{MAIN_SUMMARY["conformance_strict"]:.1%}',
     'maximum possible': f'{_ms:.1%}',
     'note': 'Rule-tagged tickets whose code no rule claims can never conform.'},
    {'metric': 'conformance loose (primary or also-maps-to)',
     'value': f'{MAIN_SUMMARY["conformance_loose"]:.1%}',
     'maximum possible': f'{_ms:.1%}', 'note': ''},
    {'metric': 'matrix code coverage',
     'value': f'{MATRIX_CODE_COVERAGE:.1%}', 'maximum possible': 'n/a',
     'note': 'Share of ALL 2026 tickets whose code some rule claims as primary. A '
             'property of the matrix. NOT a ceiling on the rates above.'},
    {'metric': 'no applicable rule',
     'value': f'{MAIN_SUMMARY["rule_na_rate"]:.1%}', 'maximum possible': 'n/a',
     'note': f'Floor is the share of tickets no rule can cover ({NA_FLOOR:.1%}).'},
    {'metric': 'tickets analysed', 'value': f'{len(TAGGED):,}',
     'maximum possible': 'n/a', 'note': ''},
    {'metric': 'CAVEAT', 'value': '', 'maximum possible': '',
     'note': 'Conformance measures agreement with the matrix, not accuracy. The '
             'resolution codes are agent-assigned and themselves noisy.'},
])


### 18.4 How strongly are the three axes related?

Cramér's V is reported with the Bergsma bias correction: at 25 × 15 and 12,000 rows the
uncorrected statistic is inflated by roughly `sqrt((r-1)(c-1)/n)`, which is not negligible here.

Theil's U is the interesting one because it is **asymmetric**. `U(code | reason)` much larger than
`U(reason | code)` means the discovered rejection reasons predict the resolution code better than
the reverse — i.e. the taxonomy we discovered is finer than the code list, which is the
justification for having discovered it at all.

χ² p-values are reported and then ignored: at n=12,000 everything is significant. Effect size is
what matters.

In [ ]:
from scipy.stats import chi2_contingency


def _entropy(counts) -> float:
    p = np.asarray(counts, dtype=float)
    p = p[p > 0]
    p = p / p.sum()
    return float(-(p * np.log(p)).sum())


def cramers_v(ct: pd.DataFrame) -> tuple:
    """Cramer's V, raw and Bergsma bias-corrected, plus the chi-square p-value."""
    chi2, p, _dof, _exp = chi2_contingency(ct.to_numpy())
    n = ct.to_numpy().sum()
    r, k = ct.shape
    phi2 = chi2 / n
    v_raw = np.sqrt(phi2 / max(1, min(r - 1, k - 1)))
    phi2c = max(0.0, phi2 - (k - 1) * (r - 1) / (n - 1))
    rc = r - (r - 1) ** 2 / (n - 1)
    kc = k - (k - 1) ** 2 / (n - 1)
    v_corr = np.sqrt(phi2c / max(1e-12, min(rc - 1, kc - 1)))
    return float(v_raw), float(v_corr), float(p)


def theils_u(x: pd.Series, y: pd.Series) -> float:
    """U(x | y): the fraction of x's entropy explained by knowing y. Asymmetric."""
    hx = _entropy(x.value_counts())
    if hx == 0:
        return float('nan')
    hxy = sum((len(g) / len(x)) * _entropy(g.value_counts())
              for _, g in x.groupby(y, observed=True))
    return float((hx - hxy) / hx)


def normalised_mi(x: pd.Series, y: pd.Series) -> float:
    hx, hy = _entropy(x.value_counts()), _entropy(y.value_counts())
    hxy = sum((len(g) / len(x)) * _entropy(g.value_counts())
              for _, g in x.groupby(y, observed=True))
    mi = hx - hxy
    return float(mi / max(1e-12, np.sqrt(hx * hy)))


PAIRS = [('rejection reason', 'reason_label', 'resolution code', RC_LABEL),
         ('validation rule',  'rule_label',   'resolution code', RC_LABEL),
         ('rejection reason', 'reason_label', 'validation rule', 'rule_label')]

_rows = []
for _an, _ac, _bn, _bc in PAIRS:
    _ct = pd.crosstab(TAGGED[_ac], TAGGED[_bc])
    _v, _vc, _p = cramers_v(_ct)
    _rows.append({
        'pair': f'{_an} x {_bn}', 'shape': f'{_ct.shape[0]}x{_ct.shape[1]}',
        'cramers_v': _v, 'v_corrected': _vc, 'nmi': normalised_mi(TAGGED[_ac], TAGGED[_bc]),
        f'U(B|A)': theils_u(TAGGED[_bc], TAGGED[_ac]),
        f'U(A|B)': theils_u(TAGGED[_ac], TAGGED[_bc]),
        'chi2_p': _p})
ASSOCIATION = pd.DataFrame(_rows)
print(ASSOCIATION.to_string(index=False, formatters={
    'cramers_v': '{:.3f}'.format, 'v_corrected': '{:.3f}'.format, 'nmi': '{:.3f}'.format,
    'U(B|A)': '{:.3f}'.format, 'U(A|B)': '{:.3f}'.format, 'chi2_p': '{:.2e}'.format}))

print('\nU(B|A) = knowing A, how much of B is determined. U(A|B) is the reverse.')
_r0 = ASSOCIATION.iloc[0]
if _r0['U(B|A)'] > _r0['U(A|B)'] + 0.05:
    print(f'\n  The discovered rejection reasons predict the resolution code '
          f'({_r0["U(B|A)"]:.2f}) better than the code predicts the reason '
          f'({_r0["U(A|B)"]:.2f}).\n  The reason taxonomy is finer-grained than the 15 codes - '
          f'which is the point of having discovered it.')
elif _r0['U(A|B)'] > _r0['U(B|A)'] + 0.05:
    print(f'\n  The resolution code predicts the rejection reason ({_r0["U(A|B)"]:.2f}) better '
          f'than the reverse ({_r0["U(B|A)"]:.2f}).\n  The discovered taxonomy is coarser than '
          f'the code list - it may be collapsing distinctions the codes make.')
else:
    print('\n  The two are close to symmetric: the discovered taxonomy is about as granular as '
          'the code list.')
print('\nchi2 p-values are all vanishingly small at this n and carry no information. '
      'Read the effect sizes.')

### 18.5 Rejection reason × validation rule — where the matrix has holes

In [ ]:
REASON_X_RULE = pd.crosstab(TAGGED['reason_label'], TAGGED['rule_label'])
REASON_X_CODE = pd.crosstab(TAGGED['reason_label'], TAGGED[RC_LABEL])

_rows = []
for _label, g in TAGGED.groupby('reason_label'):
    _na = g['rule_is_na'].mean()
    _top_rule = g.loc[~g['rule_is_na'], 'rule_label']
    _rows.append({
        'rejection_reason': _label, 'n': len(g), 'share_of_run': len(g) / len(TAGGED),
        'na_rate': _na,
        'modal_rule': _top_rule.value_counts().index[0][:50] if len(_top_rule) else '(none)',
        'modal_code': g[RC_LABEL].value_counts().index[0],
        'n_distinct_rules': int(g.loc[~g['rule_is_na'], f'{STUB_RULE}_code'].nunique()),
    })
REASON_SUMMARY = pd.DataFrame(_rows).sort_values('n', ascending=False)
print(REASON_SUMMARY.to_string(index=False, formatters={
    'share_of_run': '{:.1%}'.format, 'na_rate': '{:.0%}'.format}))

GAPS = REASON_SUMMARY[(REASON_SUMMARY['na_rate'] > 0.70) & (REASON_SUMMARY['n'] >= 30)]
print(f'\n\n{"=" * 78}\nMATRIX GAPS: rejection reasons where over 70% of tickets match no rule\n'
      f'{"=" * 78}')
if GAPS.empty:
    print('None. Every discovered rejection reason maps to a rule most of the time.')
else:
    _covered = int(GAPS['n'].sum())
    print(f'{len(GAPS)} reason(s), {_covered:,} tickets ({_covered / len(TAGGED):.1%} of the '
          f'run). Section 16 drafted candidate rules for these.\n')
    for _, r in GAPS.iterrows():
        print(f'  {r["n"]:>5} tickets ({r["na_rate"]:.0%} unmatched)  {r["rejection_reason"]}')
        print(f'        agents log these as: {r["modal_code"]}')

print('\n\nRejection reason x validation rule (counts, rules as columns):\n')
_rxr = REASON_X_RULE.copy()
_rxr.columns = [c[:22] for c in _rxr.columns]
print(_rxr.to_string())

### 18.6 The three-way table

In [ ]:
THREE_WAY = (TAGGED.groupby(['reason_label', 'rule_label', RC_LABEL], observed=True)
             .agg(n=(ID_COL, 'size'),
                  conf_strict=('conf_strict', 'mean'),
                  mappable=('is_mappable', 'first'))
             .reset_index()
             .sort_values('n', ascending=False))
THREE_WAY['share'] = THREE_WAY['n'] / len(TAGGED)

_shown = THREE_WAY[THREE_WAY['n'] >= 20]
print(f'{len(THREE_WAY):,} distinct (reason, rule, code) combinations; '
      f'{len(_shown)} with n >= 20, covering {_shown["n"].sum() / len(TAGGED):.1%} of the run.\n')
print(_shown.head(40).to_string(index=False, formatters={
    'conf_strict': '{:.0%}'.format, 'share': '{:.2%}'.format}))
print(f'\n(The full table, all {len(THREE_WAY):,} rows, goes to the three_way sheet of the '
      f'deliverable.)')

### 18.7 Disagreements — read these before quoting any number

In [ ]:
dis = TAGGED[(~TAGGED['rule_is_na']) & TAGGED['is_mappable'] & (~TAGGED['conf_strict'])]
print(f'{len(dis):,} mappable tickets where the model rule and the agent code disagree '
      f'({len(dis) / max(1, int(TAGGED["is_mappable"].sum())):.1%} of mappable).\n')

DISAGREEMENTS = dis.sample(n=min(30, len(dis)), random_state=RANDOM_SEED)[[
    ID_COL, f'{STUB_RULE}_code', 'rule_label', 'expected_primary', 'rc_actual',
    f'{STUB_RULE}_justification', f'{STUB_RULE}_evidence',
    f'{STUB_REASON}_code', 'reason_label', f'{STUB_REASON}_justification']]

for _, r in DISAGREEMENTS.head(10).iterrows():
    print('=' * 78)
    print(f'{r[ID_COL]}')
    print(f'  model rule   : {r["rule_label"]}')
    print(f'  expects code : {r["expected_primary"]}')
    print(f'  agent logged : {r["rc_actual"]}')
    print(f'  rule why     : {r[f"{STUB_RULE}_justification"]}')
    print(f'  rule evidence: {str(r[f"{STUB_RULE}_evidence"])[:220]}')
    print(f'  reason       : {r["reason_label"]}')
    print(f'  reason why   : {r[f"{STUB_REASON}_justification"]}')

print('\n' + '=' * 78)
print('For each of these, decide: is the model wrong, is the agent wrong, or is the MATRIX '
      'wrong?\nAll three happen. Conformance cannot distinguish them; only reading can.')

### 18.8 Re-run the prompt audits on the full run

Section 13 measured the three diagnostics on 1,002 probe tickets, where the 95% confidence
interval on a rate over the mappable subset is about ±3pp — wide enough that a 2pp movement
between probes means nothing. On 12,000 tickets it narrows to roughly ±0.8pp, so these are the
numbers to quote and the ones to act on if a cue still needs work.

In [ ]:
MAIN_AUDITS = run_prompt_audits(TAGGED, f'main run, {len(TAGGED):,} tickets')
PHRASE_AUDIT = MAIN_AUDITS['phrase']
FALSE_POSITIVE_AUDIT = MAIN_AUDITS['false_positive']
BOUNDARY_AUDIT = MAIN_AUDITS['boundary']


## 19. Write the deliverable

One workbook, ten sheets. Cells are truncated to Excel's 32,767-character limit; the untruncated
per-ticket text goes to the companion CSV.

The `conformance` sheet carries a `maximum possible` column next to every rate, because the
headline and the matrix's code-coverage figure can look similar and mean entirely different
things. Row 1 is the headline with its true maximum of 100%.

In [ ]:
XL_LIMIT = 32_767


def _clip(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if out[c].dtype == object:
            out[c] = out[c].map(
                lambda v: v[:XL_LIMIT - 20] + ' ...[TRUNCATED]' if isinstance(v, str)
                and len(v) > XL_LIMIT else v)
    return out


TICKET_COLS = [ID_COL, RC_LABEL, CR1_LABEL, CR2_LABEL,
               f'{STUB_RULE}_code', 'rule_label', f'{STUB_RULE}_applies',
               f'{STUB_RULE}_justification', f'{STUB_RULE}_evidence',
               f'{STUB_REASON}_code', 'reason_label',
               f'{STUB_REASON}_justification', f'{STUB_REASON}_evidence',
               'expected_primary', 'rc_actual', 'is_mappable',
               'conf_strict', 'conf_loose', 'rule_is_na', 'reason_is_none']
TICKET_COLS = [c for c in TICKET_COLS if c in TAGGED.columns]

SHEETS = {
    'tickets':        TAGGED[TICKET_COLS],
    'conformance':    CONFORMANCE_TABLE,
    'coverage':       COVERAGE_TABLE,
    'per_rule':       PER_RULE,
    'per_code':       PER_CODE,
    'rule_x_code':    RULE_X_CODE.reset_index(),
    'reason_x_code':  REASON_X_CODE.reset_index(),
    'reason_x_rule':  REASON_X_RULE.reset_index(),
    'reason_summary': REASON_SUMMARY,
    'three_way':      THREE_WAY,
    'association':    ASSOCIATION,
    'unmapped_codes': CODE_COVERAGE,
    'phrase_audit':   PHRASE_AUDIT,
    'disagreements':  DISAGREEMENTS,
}
if 'PROPOSED_RULES' in dir() and len(PROPOSED_RULES):
    SHEETS['proposed_rules'] = PROPOSED_RULES

with pd.ExcelWriter(FINAL_XLSX, engine='openpyxl') as w:
    for _name, _df in SHEETS.items():
        _clip(_df).to_excel(w, sheet_name=_name[:31], index=False)

TAGGED[TICKET_COLS].to_csv(FINAL_CSV, index=False)

print(f'Wrote {FINAL_XLSX.relative_to(BASE)}')
for _name, _df in SHEETS.items():
    print(f'  {_name:<18} {len(_df):>7,} rows x {len(_df.columns):>3} cols')
print(f'\nWrote {FINAL_CSV.relative_to(BASE)} (untruncated per-ticket text)')
print(f'\nHEADLINE: {MAIN_SUMMARY["conformance_mappable"]:.1%} conformance among mappable '
      f'tickets, against a {MATRIX_CODE_COVERAGE:.1%} ceiling.')

## Afterwards — restoring the ScienceDirect modules

`generated/pipeline_config.json` now points at the three CSD modules. The ScienceDirect ones are
still on disk (`generated/modules/sd_*.py`, `generated/taxonomies/sd_*.json`) and their outputs
are untouched under `runs/output/sd_2026_*` — only the config entry was replaced.

To put ScienceDirect back:

```
rca build --input definitions/sd_2026_pipeline_definition.xlsx --force-all
```

To come back here afterwards:

```
rca build --input definitions/csd_2026_pipeline_definition.xlsx --force-all
```

Both commands and the backed-up files are in `generated/_backup_pre_csd2026/RESTORE.txt`.

### Re-running after an edit

- **Edited a rule description or the precedence brief?** Re-run Section 9 onward. The fingerprint
  changes, the checkpoint guard discards the stale results, and the tagging repeats. Discovery
  does not.
- **Want a different rejection-reason taxonomy?** Set `REASON_TAXONOMY_OVERRIDE` in Section 8 and
  re-run from there. Discovery does not repeat.
- **Just interrupted?** Re-run the same cell. The fingerprints match, the checkpoint is honoured,
  and only the outstanding batches are dispatched — you will see `todo=` a smaller number.